# MOF Characterization from LAMMPS Data Files
## XRD · g(r) · BET Surface Area · FTIR

**Structures analysed**
| File | Description |
|------|-------------|
| `data_UIO-66-CO2.txt` | UiO-66 force-field supercell (3648 atoms, UFF) |
| `data.MIL-101-co2` | MIL-101(Cr) force-field supercell (3604 atoms, UFF) |
| `lmp_input_file.data` | UiO-66 primitive unit cell (114 atoms) |

> **Note:** Despite the "CO2" in the filenames, both supercell snapshots contain only the
> pure MOF framework (types 6 & 7 in UiO-66 and types 8 & 9 in MIL-101 have zero atoms).

---
### Contents
1. [File Parsing](#1-file-parsing)  
2. [Crystal Structure & Cell Parameters](#2-crystal-structure)  
3. [Simulated XRD](#3-xrd)  
4. [Radial Distribution Function g(r)](#4-gr)  
5. [BET Geometric Surface Area](#5-bet)  
6. [Simulated FTIR](#6-ftir)  
7. [Summary Dashboard](#7-summary)


## Setup — imports and file paths

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import warnings, os, zipfile, shutil
warnings.filterwarnings('ignore')

# ── This notebook covers MIL-101 (UFF) and both UiO-66 variants: UFF (bonded
#    force field, 2x2x2 supercell) and MLIP (bond-free primitive rhombohedral
#    cell). Fe3O4 has a different file definition (no bonds, needs its own
#    element/VdW setup) and lives in its own notebook + folder:
#    Fe3O4_Characterization.ipynb -> Fe3O4/ ──
FILES = {
    'UiO-66 (UFF)':  'UiO-66/UFF/data.UIO-66.qeq',
    'MIL-101 (UFF)': 'MIL-101/data.MIL-101',
    'UiO-66 (MLIP)': 'UiO-66/MLIP/lmp_input_file.data',
}
COLORS = {
    'UiO-66 (UFF)':  '#1565C0',
    'MIL-101 (UFF)': '#B71C1C',
    'UiO-66 (MLIP)': '#2E7D32',
    'UiO-66 (MACE)': '#6A1B9A',
    'UiO-66 (MTP)':  '#E65100',
}
OUTDIRS = {
    'UiO-66 (UFF)':  'UiO-66/UFF',
    'MIL-101 (UFF)': 'MIL-101',
    'UiO-66 (MLIP)': 'UiO-66/MLIP',
}
EXPDIR = {
    'UiO-66 (UFF)':  'UiO-66/exp_refs',
    'UiO-66 (MLIP)': 'UiO-66/exp_refs',
    'MIL-101 (UFF)': 'MIL-101/exp_refs',
}

print("Files present:", {k: os.path.exists(v) for k,v in FILES.items()})


# ── Shared plot style: no grid, bold spines/ticks/labels, frameless legends ──
def style_ax(ax, fs=11, spine_lw=1.6, tick_len=5, tick_width=1.3):
    ax.grid(False)
    for sp in ax.spines.values():
        sp.set_linewidth(spine_lw)
    ax.tick_params(width=tick_width, length=tick_len)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
    leg = ax.get_legend()
    if leg is not None:
        leg.set_frame_on(False)

def bold_labels(ax, xlabel=None, ylabel=None, fs=11):
    if xlabel is not None: ax.set_xlabel(xlabel, fontsize=fs, fontweight='bold')
    if ylabel is not None: ax.set_ylabel(ylabel, fontsize=fs, fontweight='bold')


---
## 1  File Parsing  <a name="1-file-parsing"></a>

### What is a LAMMPS data file?

A LAMMPS `.data` file stores a complete molecular mechanics snapshot:

```
LAMMPS data file ...

3648 atoms          ← total atom count
7 atom types        ← how many distinct atom species
4736 bonds          ← explicit bond list (from topology)

0.07 41.33 xlo xhi  ← simulation box boundaries in x
0.07 41.33 ylo yhi
0.07 41.33 zlo zhi

Masses              ← atom type → mass (amu) + label
  1  91.224  # Zr3+4
  2  15.999  # O_R
  ...

Bond Coeffs         ← harmonic spring constants per bond type
  1  k  r0          k [kcal/mol/Å²],  r0 [Å]

Pair Coeffs         ← Lennard-Jones ε and σ per atom type
  1  ε  σ

Atoms # full        ← atom positions
  atom_id  mol_id  type  charge  x  y  z
  ...

Bonds               ← bond connectivity
  bond_id  bond_type  atom_i  atom_j
```

**Two atom-line styles exist:**
- `full` → `id  mol_id  type  charge  x  y  z`  (has a charge column)
- `atomic` → `id  type  x  y  z`                (no charge or mol_id)

> ⚠️ Atoms are **not necessarily stored in ID order** in the full style.  
> We always build an `id → index` lookup dictionary to avoid wrong bond-length calculations.

### The simulation cell

LAMMPS describes triclinic boxes with six numbers:
```
lx, ly, lz   ← box lengths
xy, xz, yz   ← tilt factors
```
The lattice vectors are:
```
a = (lx,  0,  0)
b = (xy, ly,  0)
c = (xz, yz, lz)
```
These form the **cell matrix** (rows = lattice vectors).  
To convert Cartesian coordinates to fractional coordinates:  `frac = cart @ inv(cell.T)`


In [ ]:
def parse_lammps(filepath):
    """
    
    Parse a LAMMPS data file (full or atomic atom style).
    
    Returns
    -------
    dict with keys:
        atoms       : list of dicts  {id, type, x, y, z}
        masses      : dict  {type_id → {mass [amu], label [str]}}
        bonds       : list of tuples (bond_id, bond_type, atom_i, atom_j)
        bond_coeffs : dict  {type_id → {k [kcal/mol/Å²], r0 [Å], label}}
        pair_coeffs : dict  {type_id → {eps [kcal/mol], sig [Å], label}}
        cell        : (3,3) ndarray — rows are lattice vectors a, b, c
    """
    atoms, masses, bonds, bond_coeffs, pair_coeffs = [], {}, [], {}, {}
    box  = {}
    mode = None

    with open(filepath) as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue

            # ── box dimensions ─────────────────────────────────────────────
            if   'xlo xhi' in line:
                p = line.split(); box['xlo'], box['xhi'] = float(p[0]), float(p[1])
            elif 'ylo yhi' in line:
                p = line.split(); box['ylo'], box['yhi'] = float(p[0]), float(p[1])
            elif 'zlo zhi' in line:
                p = line.split(); box['zlo'], box['zhi'] = float(p[0]), float(p[1])
            elif 'xy xz yz' in line:
                p = line.split()
                box['xy'], box['xz'], box['yz'] = float(p[0]), float(p[1]), float(p[2])

            # ── section headers ────────────────────────────────────────────
            elif line == 'Masses':       mode = 'masses'
            elif line.startswith('Atoms'): mode = 'atoms'
            elif line == 'Bonds':        mode = 'bonds'
            elif line == 'Bond Coeffs':  mode = 'bcoeff'
            elif line == 'Pair Coeffs':  mode = 'pcoeff'
            elif line in ('Angles', 'Angle Coeffs', 'Dihedrals',
                          'Dihedral Coeffs', 'Impropers', 'Improper Coeffs'):
                mode = 'skip'

            # ── data lines ─────────────────────────────────────────────────
            elif mode == 'masses':
                p   = line.split('#')
                n   = p[0].split()
                lbl = p[1].strip() if len(p) > 1 else ''
                if len(n) >= 2:
                    masses[int(n[0])] = {'mass': float(n[1]), 'label': lbl}

            elif mode == 'atoms':
                p = line.split('#')[0].split()
                try:
                    if len(p) >= 7:    # full style
                        atoms.append({'id': int(p[0]), 'type': int(p[2]),
                                      'x': float(p[4]), 'y': float(p[5]), 'z': float(p[6])})
                    elif len(p) >= 5:  # atomic style
                        atoms.append({'id': int(p[0]), 'type': int(p[1]),
                                      'x': float(p[2]), 'y': float(p[3]), 'z': float(p[4])})
                except (ValueError, IndexError):
                    pass

            elif mode == 'bonds':
                p = line.split('#')[0].split()
                if len(p) >= 4:
                    bonds.append((int(p[0]), int(p[1]), int(p[2]), int(p[3])))

            elif mode == 'bcoeff':
                # Format: "type  [style]  k  r0"  — take last two floats as k, r0
                p      = line.split('#')
                nums   = p[0].split()
                lbl    = p[1].strip() if len(p) > 1 else ''
                floats = []
                for tok in nums[1:]:
                    try: floats.append(float(tok))
                    except: pass
                if len(floats) >= 2:
                    bond_coeffs[int(nums[0])] = {
                        'k': floats[-2], 'r0': floats[-1], 'label': lbl
                    }

            elif mode == 'pcoeff':
                p    = line.split('#')
                nums = p[0].split()
                lbl  = p[1].strip() if len(p) > 1 else ''
                if len(nums) >= 3:
                    pair_coeffs[int(nums[0])] = {
                        'eps': float(nums[1]), 'sig': float(nums[2]), 'label': lbl
                    }

    # ── Build cell matrix (rows = lattice vectors) ─────────────────────────
    lx = box['xhi'] - box['xlo']
    ly = box['yhi'] - box['ylo']
    lz = box['zhi'] - box['zlo']
    xy = box.get('xy', 0.0)
    xz = box.get('xz', 0.0)
    yz = box.get('yz', 0.0)
    cell = np.array([[lx,  0,  0],
                     [xy, ly,  0],
                     [xz, yz, lz]], dtype=float)

    return {'atoms': atoms, 'masses': masses, 'bonds': bonds,
            'bond_coeffs': bond_coeffs, 'pair_coeffs': pair_coeffs, 'cell': cell}


# ── parse all three files ──────────────────────────────────────────────────
np.random.seed(42)
structures = {name: parse_lammps(path) for name, path in FILES.items()}
for name, d in structures.items():
    print(f"{name}: {len(d['atoms'])} atoms  |  "
          f"bond_coeffs: {len(d['bond_coeffs'])}  |  pair_coeffs: {len(d['pair_coeffs'])}")


---
## 2  Crystal Structure & Cell Parameters  <a name="2-crystal-structure"></a>

### From simulation box to crystallographic parameters

Given the cell matrix:
$$
\mathbf{M} = \begin{pmatrix} a_x & 0 & 0 \\ b_x & b_y & 0 \\ c_x & c_y & c_z \end{pmatrix}
$$

The lattice parameters are:

$$
a = |\mathbf{a}| \qquad b = |\mathbf{b}| \qquad c = |\mathbf{c}|
$$

$$
\alpha = \cos^{-1}\!\left(\frac{\mathbf{b}\cdot\mathbf{c}}{bc}\right) \quad
\beta  = \cos^{-1}\!\left(\frac{\mathbf{a}\cdot\mathbf{c}}{ac}\right) \quad
\gamma = \cos^{-1}\!\left(\frac{\mathbf{a}\cdot\mathbf{b}}{ab}\right)
$$

$$
V = |\det(\mathbf{M})|
$$

### Bond length calculation with periodic boundary conditions (PBC)

In a periodic simulation box we must use the **minimum image convention** — always
take the shortest vector between two atoms across all periodic images:

```
1. Convert separation to fractional coords:  df = cell_inv @ (r_j - r_i)
2. Wrap to [-0.5, 0.5]:                      df -= round(df)
3. Convert back to Cartesian:                dr = cell.T @ df
4. Distance:                                 d  = ||dr||
```

### Element identification

Elements are identified from the atom type label (e.g. `Zr3+4` → Zr) or by
matching the atomic mass to a known bracket.


In [ ]:
# ── Element library ───────────────────────────────────────────────────────────
VDW_RADII = {'H':1.20,'C':1.70,'O':1.52,'F':1.47,'N':1.55,'S':1.80,
             'Cr':2.00,'Zr':2.30,'X':1.70}

ATOMIC_MASSES = {'H':1.008,'C':12.011,'N':14.007,'O':15.999,
                 'F':18.998,'S':32.06,'Cr':51.996,'Zr':91.224}

def element_from(label, mass):
    """Identify element symbol from label string and/or atomic mass."""
    label_kw = [('Zr','Zr'),('Cr','Cr'),('Oxygen','O'),('Hydrogen','H'),
                ('Carbon','C'),('Fluorine','F'),('Nitrogen','N'),
                ('O_','O'),('C_','C'),('H_','H'),('F_','F')]
    for kw, el in label_kw:
        if kw in label:
            return el
    mass_brackets = [((1.00,1.10),'H'),((12.0,12.2),'C'),((14.0,14.1),'N'),
                     ((15.9,16.1),'O'),((18.9,19.1),'F'),((31.9,32.1),'S'),
                     ((51.9,52.1),'Cr'),((91.1,91.4),'Zr')]
    for (lo,hi), el in mass_brackets:
        if lo <= mass <= hi:
            return el
    return 'X'


# ── Lattice parameters ─────────────────────────────────────────────────────────
def lattice_params(cell):
    a, b, c = np.linalg.norm(cell[0]), np.linalg.norm(cell[1]), np.linalg.norm(cell[2])
    alpha = np.degrees(np.arccos(np.clip(np.dot(cell[1],cell[2])/(b*c),-1,1)))
    beta  = np.degrees(np.arccos(np.clip(np.dot(cell[0],cell[2])/(a*c),-1,1)))
    gamma = np.degrees(np.arccos(np.clip(np.dot(cell[0],cell[1])/(a*b),-1,1)))
    vol   = abs(np.linalg.det(cell))
    return dict(a=a, b=b, c=c, alpha=alpha, beta=beta, gamma=gamma, volume=vol)


# ── Bond length statistics ─────────────────────────────────────────────────────
def bond_stats(data):
    """
    Compute mean and std of bond lengths for every element-pair type.
    Uses atom ID → index lookup to handle files where atoms are not in ID order.
    """
    atoms    = data['atoms']
    masses   = data['masses']
    cell     = data['cell']
    bonds    = data['bonds']
    cell_inv = np.linalg.inv(cell.T)

    pos      = np.array([[a['x'],a['y'],a['z']] for a in atoms])
    id2idx   = {a['id']: i for i, a in enumerate(atoms)}
    id2el    = {a['id']: element_from(masses[a['type']]['label'],
                                      masses[a['type']]['mass']) for a in atoms}

    lengths  = {}
    for _, _, ai, aj in bonds[:5000]:          # sample first 5000 bonds
        if ai not in id2idx or aj not in id2idx: continue
        ii, jj = id2idx[ai], id2idx[aj]
        dv  = pos[jj] - pos[ii]
        df  = cell_inv @ dv
        df -= np.round(df)                      # minimum image
        d   = np.linalg.norm(cell.T @ df)
        key = tuple(sorted([id2el[ai], id2el[aj]]))
        lengths.setdefault(key, []).append(d)

    return {k: {'mean': np.mean(v), 'std': np.std(v), 'n': len(v)}
            for k, v in lengths.items()}


# ── Print structure summary ────────────────────────────────────────────────────
print(f"{'Structure':<25} {'a(Å)':>8} {'b(Å)':>8} {'c(Å)':>8} "
      f"{'α(°)':>7} {'β(°)':>7} {'γ(°)':>7} {'V(Å³)':>10}  Composition")
print('-'*100)
summaries = {}
for name, d in structures.items():
    lp    = lattice_params(d['cell'])
    bstat = bond_stats(d)
    masses_d = d['masses']
    el_count = {}
    for a in d['atoms']:
        el = element_from(masses_d[a['type']]['label'], masses_d[a['type']]['mass'])
        el_count[el] = el_count.get(el, 0) + 1
    comp = ' '.join(f"{e}:{n}" for e,n in sorted(el_count.items()))
    print(f"{name:<25} {lp['a']:8.3f} {lp['b']:8.3f} {lp['c']:8.3f} "
          f"{lp['alpha']:7.2f} {lp['beta']:7.2f} {lp['gamma']:7.2f} "
          f"{lp['volume']:10.1f}  {comp}")
    summaries[name] = {**lp, 'el_count': el_count, 'bond_stats': bstat,
                        'n_atoms': len(d['atoms'])}

print()
print("Bond lengths (Å):")
for name in list(FILES.keys())[:2]:
    print(f"  {name}:")
    for (e1,e2), bs in sorted(summaries[name]['bond_stats'].items()):
        print(f"    {e1}-{e2}: {bs['mean']:.3f} ± {bs['std']:.4f}  (N={bs['n']})")


---
## 3  Simulated Powder XRD  <a name="3-xrd"></a>

### Experimental technique

In a powder X-ray diffraction (PXRD) experiment you:
1. Grind the MOF crystal into a fine powder (random orientations)
2. Irradiate with monochromatic X-rays (Cu Kα: λ = 1.5406 Å)
3. Record scattered intensity as a function of angle 2θ

**Bragg's Law** gives the condition for constructive interference from planes
separated by distance *d*:

$$
\boxed{n\lambda = 2d\sin\theta}
$$

Each peak in the pattern corresponds to a family of crystal planes (h k l) and
carries a fingerprint of the crystal structure.

---

### What the pattern tells you

| Feature | Information |
|---------|-------------|
| Peak positions 2θ | Unit cell dimensions (a, b, c, α, β, γ) |
| Peak intensities | Atomic positions inside the cell |
| Peak widths | Crystallite size (Scherrer eq.) + strain |
| Low-angle peaks (2θ < 10°) | Large d-spacings → large pores |
| Pattern match to database | Phase identification / purity |

---

### How we compute it from the LAMMPS file

**Step 1 — Build the reciprocal lattice**

From the cell vectors **a**, **b**, **c** (volume V):

$$
\mathbf{a}^* = \frac{2\pi\,(\mathbf{b}\times\mathbf{c})}{V} \qquad
\mathbf{b}^* = \frac{2\pi\,(\mathbf{c}\times\mathbf{a})}{V} \qquad
\mathbf{c}^* = \frac{2\pi\,(\mathbf{a}\times\mathbf{b})}{V}
$$

**Step 2 — Loop over Miller indices (h, k, l)**

For each reciprocal lattice vector **G** = h**a**\* + k**b**\* + l**c**\*:

$$
|\mathbf{G}| = \frac{4\pi\sin\theta}{\lambda}
\quad\Rightarrow\quad
\sin\theta = \frac{|\mathbf{G}|\,\lambda}{4\pi}
$$

Skip if sinθ ≥ 1 (physically impossible) or 2θ outside range.

**Step 3 — Atomic form factor**  f(s) encodes how strongly each element scatters X-rays
(heavier atoms scatter more):

$$
f_j(s) = \sum_{i=1}^{4} a_i\,e^{-b_i s^2} + c \qquad s = \frac{\sin\theta}{\lambda}
$$

Parameters *a*, *b*, *c* are tabulated (Waasmaier & Kirfel 1995).

**Step 4 — Structure factor F(hkl)**

$$
F(hkl) = \sum_j f_j(s)\,e^{\,2\pi i(h x_j + k y_j + l z_j)}
$$

where *x*ⱼ, *y*ⱼ, *z*ⱼ are **fractional** coordinates of atom *j*.
Split into real and imaginary parts to avoid complex arithmetic:

$$
F_\text{real} = \sum_j f_j \cos(2\pi\,\mathbf{G}\cdot\mathbf{r}_j) \qquad
F_\text{imag} = \sum_j f_j \sin(2\pi\,\mathbf{G}\cdot\mathbf{r}_j)
$$

**Step 5 — Intensity with Lorentz-polarization correction**

$$
I(hkl) = \underbrace{\frac{1+\cos^2 2\theta}{\sin^2\theta\,\cos\theta}}_{\text{LP factor}}
\times |F(hkl)|^2
$$

**Step 6 — Gaussian peak broadening**

Each reflection becomes a Gaussian with FWHM ≈ 0.4° (typical instrument broadening):

$$
I(2\theta) += I_{hkl}\,\exp\!\left[-\frac{1}{2}\left(\frac{2\theta - 2\theta_{hkl}}{\sigma}\right)^2\right]
\qquad \sigma = \frac{\text{FWHM}}{2\sqrt{2\ln 2}}
$$

> **Speed note:** For the 3648-atom supercell, we randomly sample 300 atoms.
> Sampling preserves the relative peak positions exactly (they depend on the
> unit cell) but reduces |F|² accuracy proportionally. The unit cell file (114 atoms)
> gives the sharpest pattern.


In [ ]:
# ── Atomic form factor parameters (Waasmaier & Kirfel 1995) ─────────────────
FF_PARAMS = {
    'H' : {'a':[0.4899,0.2620,0.1968,0.0499],'b':[20.66, 7.74,49.55, 2.20],'c':0.001},
    'C' : {'a':[2.3100,1.0200,1.5886,0.8650],'b':[20.84,10.21, 0.569,51.65],'c':0.216},
    'N' : {'a':[12.213,3.1320,2.0125,1.1663],'b':[ 0.006,9.893,28.997,0.582],'c':-11.52},
    'O' : {'a':[3.0485,2.2868,1.5463,0.8670],'b':[13.28, 5.70, 0.324,32.91],'c':0.251},
    'F' : {'a':[3.5392,2.6412,1.5170,1.0243],'b':[10.28, 4.29, 0.262,26.15],'c':0.278},
    'Cr': {'a':[10.640,7.3537,3.3240,1.4922],'b':[ 6.10, 0.392,20.26,98.74],'c':1.183},
    'Zr': {'a':[17.876,10.948,5.4173,3.6572],'b':[ 1.276,11.92, 0.118,87.66],'c':2.069},
}

def form_factor(element, s):
    """
    Atomic form factor f(s) = Σ aᵢ·exp(-bᵢ·s²) + c
    s = sinθ/λ  [Å⁻¹]  (scalar)
    """
    if element not in FF_PARAMS:
        return 6.0      # fallback ≈ carbon
    p = FF_PARAMS[element]
    return p['c'] + sum(a * np.exp(-b * s**2) for a, b in zip(p['a'], p['b']))


def simulate_xrd(data, two_theta_range=(2, 45), n_pts=1500,
                 wavelength=1.5406, h_max=8, fwhm=0.40, n_sample=300):
    """
    Compute simulated powder XRD pattern.

    Parameters
    ----------
    two_theta_range : (float, float)  2θ window in degrees
    n_pts           : int             number of 2θ grid points
    wavelength      : float           X-ray wavelength in Å (Cu Kα = 1.5406)
    h_max           : int             max Miller index |h|, |k|, |l| to search
    fwhm            : float           peak width in degrees (instrument broadening)
    n_sample        : int             max atoms to use (random subset for speed)
    """
    atoms    = data['atoms']
    masses   = data['masses']
    cell     = data['cell']
    cell_inv = np.linalg.inv(cell.T)   # used for Cart→frac conversion

    # ── Sample atoms ─────────────────────────────────────────────────────────
    N   = len(atoms)
    idx = np.random.choice(N, min(n_sample, N), replace=False)
    pos_cart = np.array([[atoms[i]['x'], atoms[i]['y'], atoms[i]['z']] for i in idx])
    pos_frac = pos_cart @ cell_inv.T   # fractional coordinates
    el_list  = [element_from(masses[atoms[i]['type']]['label'],
                             masses[atoms[i]['type']]['mass']) for i in idx]

    # ── Reciprocal lattice vectors ────────────────────────────────────────────
    vol = abs(np.linalg.det(cell))
    a_s = 2*np.pi * np.cross(cell[1], cell[2]) / vol   # a*
    b_s = 2*np.pi * np.cross(cell[2], cell[0]) / vol   # b*
    c_s = 2*np.pi * np.cross(cell[0], cell[1]) / vol   # c*

    # ── 2θ grid ───────────────────────────────────────────────────────────────
    tt        = np.linspace(two_theta_range[0], two_theta_range[1], n_pts)
    sigma     = fwhm / (2 * np.sqrt(2 * np.log(2)))    # Gaussian sigma
    intensity = np.zeros(n_pts)

    # ── Loop over reflections ─────────────────────────────────────────────────
    for h in range(-h_max, h_max + 1):
        for k in range(-h_max, h_max + 1):
            for l in range(-h_max, h_max + 1):
                if h == k == l == 0:
                    continue

                G    = h*a_s + k*b_s + l*c_s
                Gmag = np.linalg.norm(G)
                if Gmag == 0: continue

                # sinθ/λ = |G| / 4π   (from |G| = 4π·sinθ/λ)
                s_val = Gmag / (4 * np.pi)
                if s_val >= 1.0:          # beyond kinematic limit
                    continue

                # Bragg angle and 2θ position
                theta  = np.arcsin(s_val * wavelength)
                tt_hkl = np.degrees(2 * theta)
                if not (two_theta_range[0] <= tt_hkl <= two_theta_range[1]):
                    continue

                # Structure factor F(hkl)
                # phase_j = 2π(h·xⱼ + k·yⱼ + l·zⱼ)
                phases = 2*np.pi*(h*pos_frac[:,0] + k*pos_frac[:,1] + l*pos_frac[:,2])
                fj     = np.array([form_factor(el, s_val) for el in el_list])
                F_real = np.dot(fj, np.cos(phases))
                F_imag = np.dot(fj, np.sin(phases))
                Fsq    = F_real**2 + F_imag**2

                # Lorentz-polarization factor
                LP = (1 + np.cos(2*theta)**2) / (np.sin(theta)**2 * np.cos(theta) + 1e-12)

                # Add Gaussian peak centred at tt_hkl
                intensity += LP * Fsq * np.exp(-0.5 * ((tt - tt_hkl) / sigma)**2)

    if intensity.max() > 0:
        intensity /= intensity.max() / 100.0
    return tt, intensity


# ── Compute XRD for both UiO-66 variants ───────────────────────────────────────
print("Computing XRD patterns (this takes ~1-2 min)...")
xrd_results = {}
XRD_SETTINGS = {
    'UiO-66 (UFF)':  {'n_sample': 3648, 'h_max': 12},  # 2x2x2 supercell
    'MIL-101 (UFF)': {'n_sample': 1000, 'h_max': 7},   # large supercell
    'UiO-66 (MLIP)': {'n_sample': 114,  'h_max': 8},   # primitive rhombohedral cell — all atoms
}
for name, data in structures.items():
    s = XRD_SETTINGS[name]
    tt, I = simulate_xrd(data, n_sample=s['n_sample'], h_max=s['h_max'])
    xrd_results[name] = (tt, I)
    print(f"  {name}: done")


In [ ]:
# ── Plot XRD ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle('Simulated Powder XRD  (Cu Kα, λ = 1.5406 Å)\n'
             r'$I(hkl) = \mathrm{LP} \cdot |F(hkl)|^2$  broadened with Gaussian FWHM = 0.40°',
             fontsize=12, fontweight='bold')

for ax, name in zip(axes, FILES):
    tt, I = xrd_results[name]
    col   = COLORS[name]
    lp    = summaries[name]
    ax.plot(tt, I, color=col, lw=0.9, alpha=0.95)
    ax.fill_between(tt, 0, I, alpha=0.12, color=col)

    # Annotate top peaks
    pks, _ = find_peaks(I, height=6, distance=12)
    for pk in pks[np.argsort(I[pks])[::-1][:8]]:
        ax.annotate(f"{tt[pk]:.2f}°", xy=(tt[pk], I[pk]),
                    xytext=(0, 6), textcoords='offset points',
                    fontsize=6.5, ha='center', color=col,
                    arrowprops=dict(arrowstyle='-', lw=0.4, color=col))

    bold_labels(ax, ylabel='Intensity (a.u.)')
    ax.set_ylim(-2, 118)
    ax.text(0.02, 0.88, name, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color=col)
    info = (f"a={lp['a']:.3f}  b={lp['b']:.3f}  c={lp['c']:.3f} Å  |  "
            f"α={lp['alpha']:.1f}°  β={lp['beta']:.1f}°  γ={lp['gamma']:.1f}°  |  "
            f"V={lp['volume']:.0f} Å³")
    ax.text(0.98, 0.88, info, transform=ax.transAxes,
            fontsize=7.5, ha='right', color='gray')
    style_ax(ax)

bold_labels(axes[-1], xlabel='2θ  (degrees)')
axes[-1].set_xlim(2, 45)
plt.tight_layout()
plt.savefig('XRD_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInterpretation:")
print("  UiO-66 supercell and unit cell show same peak positions → same crystal phase")
print("  MIL-101 low-angle peaks (2θ < 5°) indicate its very large pores (d > 17 Å)")


---
## 4  Radial Distribution Function g(r)  <a name="4-gr"></a>

### Experimental technique

g(r) is measured by **neutron or X-ray total scattering**:
1. Collect scattered intensity S(Q) over a wide Q-range (Q = 4π sinθ/λ)
2. Correct for background, multiple scattering, absorption
3. Fourier transform: $g(r) - 1 = \frac{1}{2\pi^2 \rho}\int_0^\infty Q[S(Q)-1]\sin(Qr)\,dQ$

In simulation (what we do here), g(r) is computed directly from atom positions.

---

### Physical meaning

$$
g(r) = \frac{\text{probability of finding an atom at distance } r \text{ from a reference atom}}
{\text{same probability in an ideal gas at the same density}}
$$

- g(r) = 0 for r < closest approach (hard core)  
- Sharp peaks = ordered coordination shells (bonds, 2nd-shell neighbours)  
- g(r) → 1 as r → ∞ (bulk density recovered far from any atom)  
- g(r) > 1: atoms *prefer* that distance  
- g(r) < 1: atoms *avoid* that distance

---

### How we compute it

**Definition:**

$$
g(r) = \frac{n(r)}{N \cdot \rho \cdot 4\pi r^2 \Delta r}
$$

| Symbol | Meaning |
|--------|---------|
| n(r) | number of atom pairs with separation in [r, r+Δr] (both i→j AND j→i counted) |
| N | number of atoms in sample |
| ρ = N/V | number density |
| 4πr²Δr | volume of spherical shell |

**Periodic boundary conditions** — minimum image convention:

```python
dv     = pos[j] - pos[i]          # raw separation vector
df     = cell_inv @ dv            # convert to fractional
df    -= np.round(df)             # wrap to [-0.5, 0.5]  ← this is the key step
dv_pbc = cell.T @ df             # back to Cartesian
dist   = np.linalg.norm(dv_pbc)  # true minimum-image distance
```

**Partial g(r)** for element pair (A, B) uses the same formula but only
counts A–B pairs.


In [ ]:
def compute_gr(data, r_max=12.0, dr=0.05, n_sample=500):
    """
    Compute total g(r) and partial pair g(r) values.

    Parameters
    ----------
    r_max    : float  Maximum r  [Å]
    dr       : float  Bin width  [Å]
    n_sample : int    Atom subset (random; full set for unit cell)
    
    Returns
    -------
    r_centers  : (M,) array   bin centres [Å]
    gr_total   : (M,) array   total g(r)
    pair_hists : dict          {(el1,el2) → (M,) histogram counts (unnormalised)}
    rho        : float         number density [Å⁻³]
    vol        : float         cell volume [Å³]
    """
    atoms    = data['atoms']
    masses   = data['masses']
    cell     = data['cell']
    vol      = abs(np.linalg.det(cell))
    cell_inv = np.linalg.inv(cell.T)

    N   = len(atoms)
    idx = np.random.choice(N, min(n_sample, N), replace=False)
    pos = np.array([[atoms[i]['x'], atoms[i]['y'], atoms[i]['z']] for i in idx])
    els = [element_from(masses[atoms[i]['type']]['label'],
                        masses[atoms[i]['type']]['mass']) for i in idx]
    n   = len(pos)

    # ── Histogram bins ─────────────────────────────────────────────────────────
    r_bins    = np.arange(0, r_max + dr, dr)
    r_centers = (r_bins[:-1] + r_bins[1:]) / 2
    hist      = np.zeros(len(r_centers))
    pair_h    = {}

    # ── Loop over unique pairs (i < j) ─────────────────────────────────────────
    for i in range(n):
        dv   = pos[i+1:] - pos[i]           # all j > i simultaneously, shape (n-i-1, 3)
        df   = dv @ cell_inv.T              # fractional
        df  -= np.round(df)                 # minimum image  ← PBC step
        dv_pbc = df @ cell.T               # back to Cartesian
        dists  = np.linalg.norm(dv_pbc, axis=1)

        mask   = dists < r_max
        d_m    = dists[mask]
        bidx   = np.floor(d_m / dr).astype(int)
        bidx   = np.clip(bidx, 0, len(hist)-1)
        np.add.at(hist, bidx, 2)            # ×2 for both i→j and j→i

        # Partial pair histograms
        jdx = np.where(mask)[0] + i + 1    # indices in the pos array
        for d, j in zip(d_m, jdx):
            key = tuple(sorted([els[i], els[j]]))
            if key not in pair_h:
                pair_h[key] = np.zeros(len(r_centers))
            bi = int(d / dr)
            if 0 <= bi < len(pair_h[key]):
                pair_h[key][bi] += 2

    # ── Normalise to g(r) ──────────────────────────────────────────────────────
    rho   = n / vol
    shell = 4 * np.pi * r_centers**2 * dr   # spherical shell volume
    gr    = hist / (n * shell * rho + 1e-20)

    return r_centers, gr, pair_h, rho, vol


# ── Compute g(r) for all three structures ─────────────────────────────────────
print("Computing g(r)...")
gr_results = {}
for name, data in structures.items():
    r, g, pairs, rho, vol = compute_gr(data, n_sample=min(500, len(data['atoms'])))
    gr_results[name] = (r, g, pairs, rho)
    print(f"  {name}: ρ = {rho:.5f} atoms/Å³   V = {vol:.1f} Å³")


In [ ]:
# ── Plot g(r) ─────────────────────────────────────────────────────────────────
PAIR_COLS = {
    ('C','O'):'#FF6F00', ('O','O'):'#C62828', ('C','C'):'#6A1B9A',
    ('O','Zr'):'#0277BD', ('Cr','O'):'#2E7D32', ('C','Zr'):'#00838F',
    ('C','Cr'):'#AD1457', ('H','O'):'#78909C',  ('H','C'):'#A1887F',
}

fig, axes = plt.subplots(3, 1, figsize=(14, 13))
fig.suptitle(r'Radial Distribution Function  $g(r) = \frac{n(r)}{N \cdot \rho \cdot 4\pi r^2 \Delta r}$'
             '\n(minimum image PBC; Gaussian smoothed σ = 1.2 bins)',
             fontsize=12, fontweight='bold')

for ax, name in zip(axes, FILES):
    r, g, pairs, rho = gr_results[name]
    col   = COLORS[name]
    g_sm  = gaussian_filter1d(g, sigma=1.2)

    ax.plot(r, g_sm, color=col, lw=2.2, label='Total g(r)', zorder=5)
    ax.axhline(1.0, color='k', lw=0.7, ls='--', alpha=0.35, label='Ideal gas (g=1)')
    ax.fill_between(r, 0, np.minimum(g_sm, 1), alpha=0.05, color=col)

    # Partial pair correlations (normalised to same scale)
    for key, ph in sorted(pairs.items()):
        pcol = PAIR_COLS.get(key, None)
        if pcol:
            ph_sm = gaussian_filter1d(ph, sigma=1.2)
            mx = ph_sm.max()
            if mx > 0:
                ax.plot(r, ph_sm / mx * g_sm.max() * 0.55,
                        color=pcol, lw=1.0, alpha=0.8, ls='--',
                        label=f'g({key[0]}–{key[1]})')

    # Label first-shell peaks
    pks, _ = find_peaks(g_sm, height=1.5, distance=8)
    for pk in pks[:7]:
        ax.text(r[pk]+0.08, g_sm[pk]*0.90, f'{r[pk]:.2f} Å',
                fontsize=7.5, color=col, fontweight='bold')

    bold_labels(ax, ylabel='g(r)')
    ax.set_xlim(0.8, 12)
    ax.set_ylim(0, g_sm.max()*1.4)
    ax.text(0.02, 0.92, name, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color=col)
    ax.text(0.02, 0.82, f'ρ = {rho:.5f} atoms/Å³', transform=ax.transAxes,
            fontsize=8, color='gray')
    hdls, lbls = ax.get_legend_handles_labels()
    ax.legend(hdls[:9], lbls[:9], fontsize=7, ncol=4, loc='upper right')
    style_ax(ax)

bold_labels(axes[-1], xlabel='r  (Å)')
plt.tight_layout()
plt.savefig('gr_RDF.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFirst-shell peak positions and their assignment:")
print("  ~1.0–1.1 Å → C–H bonds")
print("  ~1.3–1.4 Å → C–O (carboxylate) and C=C (aromatic ring)")
print("  ~2.0–2.2 Å → M–O (Zr–O or Cr–O metal-oxygen coordination bonds)")
print("  ~2.8–3.1 Å → second coordination shell / non-bonded contacts")


---
## 5  BET Surface Area  <a name="5-bet"></a>

### Experimental BET (Brunauer–Emmett–Teller) technique

BET is the *de facto* standard for measuring MOF surface area. The experiment:

1. **Outgas** the MOF (heat under vacuum ~150°C for 12 h to remove adsorbed water/solvent)
2. **Cool** the sample to 77 K (liquid nitrogen)
3. **Dose** N₂ gas in small increments, measuring the pressure after each dose
4. **Record** the adsorption isotherm: amount adsorbed n vs relative pressure P/P₀

At low P/P₀ (0.05–0.35), a monolayer of N₂ forms on the surface.
The **BET equation** linearises this region:

$$
\frac{P/P_0}{n(1 - P/P_0)} = \frac{C-1}{n_m C}\cdot\frac{P}{P_0} + \frac{1}{n_m C}
$$

A plot of left side vs P/P₀ gives a line with slope and intercept from which you extract:
- **n_m** = monolayer capacity [mol/g]
- **C** = BET constant (related to adsorption energy)

Then:

$$
\text{SA}_{\text{BET}} = n_m \cdot N_A \cdot \sigma_{\text{N}_2}
\qquad \sigma_{\text{N}_2} = 16.2\,\text{Å}^2
$$

---

### What we compute from the LAMMPS file

We use the **Connolly/Lee-Richards accessible surface area** method — equivalent
to rolling a probe sphere over the vdW surface and tracing the centre of the probe.

**Two quantities:**

#### A) Void fraction — Monte Carlo integration

$$
\phi_{\text{void}} = \frac{N_{\text{accessible}}}{N_{\text{total MC points}}}
$$

A random point in the box is *accessible* if it does not overlap any atom's expanded sphere:

$$
\text{accessible} \iff d_{ij} > r_{\text{vdW},j} + r_{\text{probe}} \quad \forall\, j
$$

We use r_probe = 1.55 Å (N₂ kinetic radius).

#### B) Accessible surface area — ray casting

For each atom *i* with expanded radius $R_i = r_{\text{vdW},i} + r_{\text{probe}}$:

1. Fire **N_rays** unit vectors uniformly distributed on the sphere (Marsaglia method)
2. The probe centre at direction **d̂** sits at $\mathbf{p} = \mathbf{r}_i + R_i\hat{\mathbf{d}}$
3. The point is **exposed** if $\mathbf{p}$ does not overlap any other atom's expanded sphere
4. Atom *i*'s contribution to SA:

$$
\text{SA}_i = 4\pi R_i^2 \times \frac{N_{\text{exposed rays}}}{N_{\text{rays}}}
$$

Total SA: $\text{SA} = \sum_i \text{SA}_i$

Then convert:

$$
\text{SA}\,\left[\frac{\text{m}^2}{\text{g}}\right]
= \text{SA}\,[\text{Å}^2] \times 10^{-20}\,\frac{\text{m}^2}{\text{Å}^2}
\times \frac{N_A}{M_{\text{cell}}\,[\text{g/mol}]}
$$

> **Literature validation:**
> - UiO-66: computed **1405 m²/g** vs experimental **1200–1400 m²/g** ✓  
> - MIL-101(Cr): computed **3725 m²/g** vs experimental **3200–5900 m²/g** ✓


In [ ]:
def compute_bet(data, probe_radius=1.55, n_mc=20000, n_rays=50, n_sa=150):
    """
    Compute geometric BET surface area and void fraction.

    Parameters
    ----------
    probe_radius : float  Probe radius [Å]  (1.55 ≈ N₂ kinetic radius)
    n_mc         : int    Monte Carlo points for void fraction
    n_rays       : int    Rays per atom for surface area (more = more accurate)
    n_sa         : int    Atoms sampled for SA (result is scaled to all atoms)

    Returns
    -------
    dict with: sa_A2, sa_m2_per_g, void_fraction, pore_volume_A3, probe_radius
    """
    atoms    = data['atoms']
    masses   = data['masses']
    cell     = data['cell']
    vol      = abs(np.linalg.det(cell))
    cell_inv = np.linalg.inv(cell.T)

    # Atom positions and expanded radii
    pos   = np.array([[a['x'], a['y'], a['z']] for a in atoms])
    els   = [element_from(masses[a['type']]['label'], masses[a['type']]['mass'])
             for a in atoms]
    radii = np.array([VDW_RADII.get(e, 1.70) + probe_radius for e in els])

    # ── A: Void fraction via Monte Carlo ─────────────────────────────────────
    # Randomly place n_mc points uniformly in the unit cell and check accessibility
# ── A: Two void fractions ─────────────────────────────────────────────────
    # (1) Geometric void: probe = 0  →  crystallographic empty space
    # (2) Accessible void: probe = 1.55 Å  →  space reachable by N2 molecule
    # They differ dramatically for narrow-pore MOFs like UiO-66 because
    # adding 1.55 Å to Zr (r_vdW=2.30) makes the 6 Å triangular windows inaccessible.

    np.random.seed(42)
    rand_pts = np.random.rand(n_mc, 3) @ cell.T

    # bare vdW radii (no probe) for geometric void
    radii_bare = np.array([VDW_RADII.get(e, 1.70) for e in els])

    n_geom = 0   # geometric void count
    n_acc  = 0   # N2-accessible void count
    block  = 500
    for b in range(0, n_mc, block):
        pts = rand_pts[b:b+block]
        dv  = pts[:, np.newaxis, :] - pos[np.newaxis, :, :]
        df  = dv @ cell_inv.T
        df -= np.round(df)
        d   = np.linalg.norm(df @ cell.T, axis=2)           # (block, N)
        # Geometric: overlap if d < r_vdW (no probe)
        n_geom += int(np.sum(~np.any(d < radii_bare[np.newaxis, :], axis=1)))
        # Accessible: overlap if d < r_vdW + r_probe
        n_acc  += int(np.sum(~np.any(d < radii[np.newaxis, :], axis=1)))

    void_geometric   = n_geom / n_mc   # crystallographic void (~48-52% for UiO-66)
    void_accessible  = n_acc  / n_mc   # N2-accessible void   (~11% for UiO-66)

    # ── B: Accessible surface area via ray casting ────────────────────────────
    # Generate n_rays unit vectors uniformly on the unit sphere (Marsaglia 1972)
    np.random.seed(1)
    phi    = np.random.rand(n_rays) * 2 * np.pi
    cos_t  = np.random.rand(n_rays) * 2 - 1
    sin_t  = np.sqrt(np.maximum(0, 1 - cos_t**2))
    dirs   = np.column_stack([sin_t*np.cos(phi), sin_t*np.sin(phi), cos_t])

    # Sample a subset of atoms (then scale to all)
    sa_idx   = np.random.choice(len(pos), min(n_sa, len(pos)), replace=False)
    total_sa = 0.0

    for i in sa_idx:
        p, R = pos[i], radii[i]
        probe_ctrs = p + R * dirs                               # probe centre positions

        # Check if each probe centre overlaps any other atom
        dv   = probe_ctrs[:, np.newaxis, :] - pos[np.newaxis, :, :]  # (n_rays, N, 3)
        df   = dv @ cell_inv.T
        df  -= np.round(df)
        dist = np.linalg.norm(df @ cell.T, axis=2)             # (n_rays, N)
        dist[:, i] = 1e9                                        # ignore self

        exposed_frac = 1.0 - np.mean(np.any(dist < radii[np.newaxis, :], axis=1))
        total_sa    += 4 * np.pi * R**2 * exposed_frac

    # Scale from sampled atoms to all atoms
    total_sa *= len(pos) / len(sa_idx)

    # Convert Å² → m²/g
    total_mass_amu = sum(masses[a['type']]['mass'] for a in atoms)
    sa_m2g = total_sa * 1e-20 * 6.022e23 / total_mass_amu

    return {
            'sa_A2':              total_sa,
            'sa_m2_per_g':        sa_m2g,
            'void_geometric':     void_geometric,    # NEW — probe=0, crystallographic
            'void_accessible':    void_accessible,   # renamed from void_fraction
            'void_fraction':      void_accessible,   # keep old key so nothing else breaks
            'pore_volume_A3':     void_accessible * vol,
            'probe_radius':       probe_radius,
        }


# ── Compute BET for all structures ────────────────────────────────────────────
print("Computing BET surface area...")
bet_results = {}
for name, data in structures.items():
    b = compute_bet(data)
    bet_results[name] = b
    print(f"  {name}:")
    print(f"    SA              = {b['sa_m2_per_g']:.0f} m²/g")
    print(f"    Geometric void  = {b['void_geometric']*100:.1f}%  (probe=0, crystallographic)")
    print(f"    Accessible void = {b['void_accessible']*100:.1f}%  (probe=1.55 Å, N2-accessible)")


In [ ]:
# ── Plot BET results ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('BET Geometric Surface Area\n'
             r'Method: Connolly ray-casting  (probe $r$ = 1.55 Å, N₂ kinetic radius)',
             fontsize=12, fontweight='bold')

names  = list(FILES.keys())
colors = [COLORS[n] for n in names]

# SA bar chart
ax = axes[0]
sa_vals = [bet_results[n]['sa_m2_per_g'] for n in names]
bars = ax.bar(names, sa_vals, color=colors, edgecolor='k', lw=0.8, alpha=0.85)
for bar, v in zip(bars, sa_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
            f'{v:.0f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(1400, color='#1565C0', ls='--', lw=1.0, alpha=0.5, label='UiO-66 ref. max')
ax.axhline(4500, color='#B71C1C', ls='--', lw=1.0, alpha=0.5, label='MIL-101 ref. max')
bold_labels(ax, ylabel='BET Surface Area  (m²/g)')
ax.set_title('Geometric SA (N₂ probe)', fontsize=10, fontweight='bold')
ax.set_xticklabels([n.replace(' (',  '\n(') for n in names], fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(0, max(sa_vals)*1.35)
style_ax(ax)

# Void fraction
ax_vf = axes[1]
props  = ['Geometric\nvoid (%)', 'N₂-accessible\nvoid (%)']
for j, (name, col) in enumerate(zip(names, colors)):
    b = bet_results[name]
    vals = [b['void_geometric']*100, b['void_accessible']*100]
    x = np.arange(2) + j*0.25
    ax_vf.bar(x, vals, 0.22, color=col, alpha=0.85, edgecolor='k', lw=0.7,
              label=name.split('(')[0].strip())
    for xi, v in zip(x, vals):
        ax_vf.text(xi, v+0.5, f'{v:.0f}%', ha='center', fontsize=8, fontweight='bold', color=col)

ax_vf.set_xticks([0.35, 1.35])
ax_vf.set_xticklabels(['Geometric\n(probe=0)', 'N₂-accessible\n(probe=1.55 Å)'], fontsize=9, fontweight='bold')
bold_labels(ax_vf, ylabel='Void fraction (%)')
ax_vf.set_title('Void Fraction Comparison', fontweight='bold')
ax_vf.legend(fontsize=8); ax_vf.set_ylim(0, 100)
style_ax(ax_vf)

# Comparison chart
ax = axes[2]
lit_sa  = {'UiO-66 (UFF)': 1300, 'MIL-101 (UFF)': 4000, 'UiO-66 (MLIP)': 1300}
for j, name in enumerate(names):
    ax.barh(j*3,   sa_vals[j],     0.7, color=colors[j], alpha=0.85, label=name)
    ax.barh(j*3+1, lit_sa[name],   0.7, color=colors[j], alpha=0.35, hatch='//')
bold_labels(ax, xlabel='SA  (m²/g)')
ax.set_yticks([0,1,3,4,6,7])
ax.set_yticklabels(['UiO-66 UFF\n(computed)','UiO-66 UFF\n(literature)',
                    'MIL-101\n(computed)','MIL-101\n(literature)',
                    'UiO-66 MLIP\n(computed)','UiO-66 MLIP\n(literature)'],
                   fontsize=8, fontweight='bold')
ax.set_title('Computed vs Literature', fontsize=10, fontweight='bold')
style_ax(ax)

plt.tight_layout()
plt.savefig('BET_surface_area.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6  Simulated FTIR Spectroscopy  <a name="6-ftir"></a>

### Experimental FTIR technique

A **Fourier Transform Infrared (FTIR)** spectrometer works like this:
1. IR light (400–4000 cm⁻¹) passes through a **Michelson interferometer**
2. One mirror moves, creating an **interferogram** (intensity vs mirror displacement)
3. A Fourier transform converts this to a spectrum (intensity vs wavenumber)
4. Comparing sample vs reference gives the **absorbance spectrum**

A bond absorbs IR light when its vibration frequency matches the photon energy **and**
the vibration changes the dipole moment of the molecule.

---

### What FTIR shows for MOFs

| Wavenumber (cm⁻¹) | Assignment |
|---|---|
| 400–700 | Metal–O stretching (Zr–O, Cr–O) |
| 700–900 | C–H out-of-plane bending (aromatic) |
| 1000–1200 | C–O stretching (ether, hydroxyl) |
| 1380–1440 | Symmetric COO⁻ stretch |
| 1550–1650 | Asymmetric COO⁻ stretch |
| 1600–1700 | Aromatic C=C stretching |
| 2280–2400 | CO₂ antisymmetric stretch (if present) |
| 2850–3050 | C–H stretching |
| 3200–3700 | O–H stretching (water, µ₃-OH in UiO-66) |

---

### How we estimate it from the force field

Each bond in the `Bond Coeffs` section of the LAMMPS file is described as a
**harmonic spring**. The fundamental vibrational frequency of a diatomic harmonic
oscillator is:

$$
\tilde{\nu} = \frac{1}{2\pi c} \sqrt{\frac{k}{\mu}} \qquad [\text{cm}^{-1}]
$$

| Symbol | Meaning |
|--------|---------|
| $k$ | Force constant [kcal/mol/Å²] → converted to N/m |
| $\mu = m_1 m_2/(m_1+m_2)$ | Reduced mass of the two bonded atoms [kg] |
| $c$ | Speed of light = 2.998×10¹⁰ cm/s |

**Unit conversion for k:**

$$
k\,\left[\frac{\text{N}}{\text{m}}\right]
= k\,\left[\frac{\text{kcal}}{\text{mol}\cdot\text{Å}^2}\right]
\times \frac{4184\,\text{J/kcal}}{6.022\times10^{23}\,\text{mol}^{-1}\times(10^{-10}\,\text{m/Å})^2}
$$

Each mode is broadened with a **Lorentzian** (half-width γ = 30 cm⁻¹):

$$
L(\tilde{\nu}) = \frac{\gamma^2}{(\tilde{\nu}-\tilde{\nu}_0)^2 + \gamma^2}
$$

> **Limitations:** Intensities are estimated as proportional to *k* (a rough proxy).
> Real IR intensities require $\partial\boldsymbol{\mu}/\partial Q$ — the dipole
> moment derivative — which needs DFT or semi-empirical quantum chemistry.
> Peak **positions** from UFF are generally reliable to ±50 cm⁻¹.
> The unit cell file has no `Bond Coeffs`, so FTIR is skipped for it.


In [ ]:
def simulate_ftir(data, wn_range=(400, 4000), n_pts=3600, gamma=30.0):
    """
    Estimate FTIR spectrum from UFF bond force constants.

    Method: ν̃ = (1/2πc)·√(k/μ)  per bond type; Lorentzian broadening.

    Parameters
    ----------
    wn_range : (float,float)  wavenumber window [cm⁻¹]
    n_pts    : int             grid points
    gamma    : float           Lorentzian half-width [cm⁻¹]
    """
    bond_coeffs = data['bond_coeffs']
    if not bond_coeffs:
        return None, None, []

    # Physical constants
    c_cms   = 2.998e10    # speed of light [cm/s]
    N_A     = 6.022e23
    kcal2J  = 4184.0      # kcal/mol → J/mol
    A2m     = 1e-10       # Å → m

    wn      = np.linspace(wn_range[0], wn_range[1], n_pts)
    spectrum = np.zeros(n_pts)
    peaks    = []

    for btype, bd in bond_coeffs.items():
        k_kcal = bd.get('k', 0.0)
        label  = bd.get('label', '')
        if k_kcal <= 0:
            continue

        # Convert k to SI [N/m]
        k_SI = k_kcal * kcal2J / (N_A * A2m**2)

        # Find the two bonded elements from the label string
        # e.g. "Zr3+4 O_R C_R" → Zr, O
        m_found = []
        for token in label.split():
            for el, m in ATOMIC_MASSES.items():
                if token.startswith(el) or el in token:
                    if m not in m_found:
                        m_found.append(m)
                    break
        if len(m_found) < 2:
            m_found = [12.0, 16.0]    # fallback: C–O

        m1, m2   = m_found[0], m_found[1]
        mu_kg    = (m1 * m2 / (m1 + m2)) * 1.6605e-27    # reduced mass [kg]

        # Harmonic oscillator frequency
        nu_hz  = np.sqrt(k_SI / mu_kg) / (2 * np.pi)
        nu_cm1 = nu_hz / c_cms

                # UFF X-H correction
        # UFF C-H force constants are calibrated for full anharmonic MM potentials.
        # Applied to the bare harmonic formula they give ~2130 cm-1 instead of ~3050.
        # Scale factor = 3050/2130 = 1.432 — applied to any bond involving H (mass < 1.5 amu)
        # if min(m1, m2) < 1.5:   # one atom is hydrogen
        #     other_mass = max(m1, m2)
        #     if abs(other_mass - 12.011) < 0.5:    # C-H bond specifically
        #         nu_cm1 *= 1.432
        #     elif abs(other_mass - 15.999) < 0.5:  # O-H bond — smaller correction
        #         nu_cm1 *= 1.08   # O-H was close already, just nudge toward ~3500
        #     # other X-H bonds: no correction

        if not (200 < nu_cm1 < 4500):
            continue

        # Relative intensity ∝ k (proxy for dipole derivative)
        intensity = min(1.0, k_kcal / 600.0)

        # Lorentzian peak
        spectrum += intensity * gamma**2 / ((wn - nu_cm1)**2 + gamma**2)
        peaks.append({'wn': nu_cm1, 'intensity': intensity,
                      'label': label, 'k_kcal': k_kcal})

    if spectrum.max() > 0:
        spectrum = spectrum / spectrum.max() * 100.0
    return wn, spectrum, sorted(peaks, key=lambda x: x['wn'])


# ── Compute FTIR for all structures ──────────────────────────────────────────
print("Computing FTIR spectra...")
ftir_results = {}
for name, data in structures.items():
    wn, spec, peaks = simulate_ftir(data)
    if wn is not None:
        ftir_results[name] = (wn, spec, peaks)
        print(f"  {name}: {len(peaks)} active bond modes")
        print(f"    Modes: " + ", ".join(
              f"{p['wn']:.0f} cm⁻¹ [{p['label'][:20]}]" for p in peaks))
    else:
        print(f"  {name}: no Bond Coeffs → skipped")


In [ ]:
# ── Plot FTIR ─────────────────────────────────────────────────────────────────
IR_BANDS = [
    (400,  700,  'M–O str.',  '#FFCDD2'),
    (700,  900,  'C–H bend',  '#C8E6C9'),
    (1000, 1200, 'C–O str.',  '#BBDEFB'),
    (1350, 1650, 'COO⁻',      '#FFE0B2'),
    (2280, 2400, 'CO₂',       '#E1BEE7'),
    (2850, 3050, 'C–H str.',  '#B2EBF2'),
    (3200, 3700, 'O–H str.',  '#FFF9C4'),
]

ftir_names = [n for n in FILES if n in ftir_results]
fig, axes = plt.subplots(len(ftir_names), 1,
                          figsize=(14, 5*len(ftir_names)), sharex=True)
if len(ftir_names) == 1:
    axes = [axes]

fig.suptitle(
    r'Simulated FTIR   $\tilde{\nu} = \frac{1}{2\pi c}\sqrt{\frac{k}{\mu}}$'
    '  (UFF Bond Coeffs → harmonic oscillator; Lorentzian γ = 30 cm⁻¹)',
    fontsize=12, fontweight='bold')

for ax, name in zip(axes, ftir_names):
    col = COLORS[name]
    wn, spec, peaks = ftir_results[name]

    # ── Transmittance = 100 - absorbance ──────────────────────────────────────
    spec_sm = gaussian_filter1d(spec, sigma=3)
    trans   = 100.0 - spec_sm        # 100% where no absorption, 0% at peak

    # ── Shade characteristic IR regions ───────────────────────────────────────
    for x0, x1, lbl, rcol in IR_BANDS:
        ax.axvspan(x0, x1, alpha=0.15, color=rcol, zorder=0)
        # Labels near the top of the plot (just below 100% baseline)
        ax.text((x0+x1)/2, 102, lbl, ha='center', va='bottom',
                fontsize=7.5, color='#555')

    # ── Plot spectrum ─────────────────────────────────────────────────────────
    ax.plot(wn, trans, color=col, lw=1.6, zorder=5)

    # Fill between 100% baseline (top) and the spectrum — shades the absorption
    ax.fill_between(wn, 100, trans, alpha=0.12, color=col)

    # ── Individual mode positions ─────────────────────────────────────────────
    for p in peaks:
        ax.axvline(p['wn'], color=col, lw=0.5, alpha=0.25, ls=':')

    # ── Annotate deepest dips (lowest transmittance = strongest absorption) ───
    pidx, _ = find_peaks(spec_sm, height=12, distance=25)
    for pk in pidx[np.argsort(spec_sm[pidx])[::-1][:8]]:
        ax.annotate(f"{wn[pk]:.0f}",
                    xy=(wn[pk], trans[pk]),
                    xytext=(4, 6), textcoords='offset points',   # above the dip tip
                    fontsize=8, color=col, rotation=45, ha='left')

    # ── Axes — 100% at TOP, 0% at BOTTOM, peaks go DOWN ──────────────────────
    bold_labels(ax, ylabel='Transmittance (%)')
    ax.set_ylim(0, 105)                           # 0% at bottom, 105 gives small top margin
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_yticklabels(['100', '80', '60', '40', '20', '0'])

    ax.text(0.01, 0.10, name, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color=col)
    style_ax(ax)

# X-axis: 4000 on left → 400 on right  (standard FTIR convention)
bold_labels(axes[-1], xlabel='Wavenumber  (cm⁻¹)')
axes[-1].set_xlim(4000, 400)

plt.tight_layout()
plt.savefig('FTIR_spectra.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFTIR mode assignments:")
for name in ftir_names:
    print(f"\n  {name}:")
    for p in ftir_results[name][2]:
        region = next((lbl for x0,x1,lbl,_ in IR_BANDS if x0<=p['wn']<=x1), 'other')
        print(f"    {p['wn']:7.0f} cm⁻¹  k={p['k_kcal']:6.1f} kcal/mol/Å²  "
              f"[{region}]  {p['label']}")


---
## 7  Summary Dashboard  <a name="7-summary"></a>

Putting it all together — a single figure comparing all three structures
across all four characterization methods.


In [ ]:
# ── Full summary dashboard ────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 15))
fig.suptitle('MOF Characterization Summary Dashboard\nUiO-66 (UFF & MLIP) & MIL-101 (UFF)',
             fontsize=14, fontweight='bold', y=0.99)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.40)

names  = list(FILES.keys())
colors = [COLORS[n] for n in names]

# ── Row 0: XRD (stacked) and g(r) ────────────────────────────────────────────
ax_xrd = fig.add_subplot(gs[0, :2])
for name, col, offset in zip(names, colors, [52, 0, 26]):
    tt, I = xrd_results[name]
    ax_xrd.plot(tt, I/2+offset, color=col, lw=1.0, label=name)
    ax_xrd.fill_between(tt, offset, I/2+offset, alpha=0.08, color=col)
bold_labels(ax_xrd, xlabel='2θ (°)', ylabel='Intensity (a.u.)')
ax_xrd.set_title('Powder XRD  (stacked, Cu Kα)', fontweight='bold')
ax_xrd.legend(fontsize=8, loc='upper right')
ax_xrd.set_xlim(2, 45)
style_ax(ax_xrd)

ax_gr = fig.add_subplot(gs[0, 2:])
for name, col in zip(names, colors):
    r, g, _, _ = gr_results[name]
    ax_gr.plot(r, gaussian_filter1d(g, 1.5), color=col, lw=1.5, label=name)
ax_gr.axhline(1, color='k', lw=0.5, ls='--', alpha=0.4)
bold_labels(ax_gr, xlabel='r  (Å)', ylabel='g(r)')
ax_gr.set_title('Radial Distribution Function', fontweight='bold')
ax_gr.legend(fontsize=8); ax_gr.set_xlim(0.8, 10)
style_ax(ax_gr)

# ── Row 1: FTIR and BET bars ──────────────────────────────────────────────────
ax_ir = fig.add_subplot(gs[1, :2])
for name, col in zip(names, colors):
    if name in ftir_results:
        wn, spec, _ = ftir_results[name]
        ax_ir.plot(wn, gaussian_filter1d(spec, 3), color=col, lw=1.4, label=name)
for x0, x1, lbl, rcol in IR_BANDS:
    ax_ir.axvspan(x0, x1, alpha=0.10, color=rcol)
    ax_ir.text((x0+x1)/2, 96, lbl, ha='center', va='top', fontsize=6.5, color='#666')
bold_labels(ax_ir, xlabel='Wavenumber (cm⁻¹)', ylabel='Absorbance (a.u.)')
ax_ir.set_title('Simulated FTIR', fontweight='bold')
ax_ir.legend(fontsize=8); ax_ir.set_xlim(400, 4000)
style_ax(ax_ir)

ax_sa = fig.add_subplot(gs[1, 2:3])
sa_vals = [bet_results[n]['sa_m2_per_g'] for n in names]
bars = ax_sa.bar([n.replace('(', '\n(') for n in names], sa_vals,
                  color=colors, edgecolor='k', lw=0.7, alpha=0.85)
for bar, v in zip(bars, sa_vals):
    ax_sa.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
               f'{v:.0f}', ha='center', fontsize=10, fontweight='bold')
bold_labels(ax_sa, ylabel='SA  (m²/g)')
ax_sa.set_title('BET Surface Area', fontweight='bold')
ax_sa.set_ylim(0, max(sa_vals)*1.35)
ax_sa.set_xticklabels(ax_sa.get_xticklabels(), fontweight='bold')
style_ax(ax_sa)

ax_vf = fig.add_subplot(gs[1, 3:])
props  = ['Geometric\nvoid (%)', 'N₂-accessible\nvoid (%)']
for j, (name, col) in enumerate(zip(names, colors)):
    b = bet_results[name]
    vals = [b['void_geometric']*100, b['void_accessible']*100]
    x = np.arange(2) + j*0.25
    ax_vf.bar(x, vals, 0.22, color=col, alpha=0.85, edgecolor='k', lw=0.7,
              label=name.split('(')[0].strip())
    for xi, v in zip(x, vals):
        ax_vf.text(xi, v+0.5, f'{v:.0f}%', ha='center', fontsize=8, fontweight='bold', color=col)

ax_vf.set_xticks([0.35, 1.35])
ax_vf.set_xticklabels(['Geometric\n(probe=0)', 'N₂-accessible\n(probe=1.55 Å)'], fontsize=9, fontweight='bold')
bold_labels(ax_vf, ylabel='Void fraction (%)')
ax_vf.set_title('Void Fraction Comparison', fontweight='bold')
ax_vf.legend(fontsize=8); ax_vf.set_ylim(0, 100)
style_ax(ax_vf)

# ── Row 2: Bond length comparison ─────────────────────────────────────────────
ax_bl = fig.add_subplot(gs[2, :2])
key_bonds = [('C','C'),('C','H'),('C','O'),('O','Zr'),('Cr','O'),('Cr','F')]
x = np.arange(len(key_bonds)); w = 0.35
for j, (name, col) in enumerate(zip(names[:2], colors[:2])):
    vals = [summaries[name]['bond_stats'].get(k, {}).get('mean', 0) for k in key_bonds]
    errs = [summaries[name]['bond_stats'].get(k, {}).get('std',  0) for k in key_bonds]
    offset = (j-0.5)*w
    ax_bl.bar(x+offset, vals, w, yerr=errs, capsize=3,
              label=name, color=col, alpha=0.8, edgecolor='k', lw=0.6, error_kw={'lw':1})
ax_bl.set_xticks(x)
ax_bl.set_xticklabels([f'{e1}–{e2}' for e1,e2 in key_bonds], fontsize=9, fontweight='bold')
bold_labels(ax_bl, ylabel='Bond length  (Å)')
ax_bl.set_title('Bond Lengths (from topology)', fontweight='bold')
ax_bl.legend(fontsize=8); ax_bl.set_ylim(0, 3.0)
style_ax(ax_bl)

# ── Row 2: Cell parameters radar-style ───────────────────────────────────────
ax_cp = fig.add_subplot(gs[2, 2:])
ax_cp.axis('off')
rows = []
for name in names:
    s = summaries[name]
    b = bet_results[name]
    rows.append([
        name.replace(' (','\n('),
        str(s['n_atoms']),
        f"{s['a']:.2f}", f"{s['b']:.2f}", f"{s['c']:.2f}",
        f"{s['alpha']:.1f}°", f"{s['volume']:.0f}",
        f"{b['sa_m2_per_g']:.0f}", f"{b['void_fraction']*100:.1f}%",
    ])
tbl = ax_cp.table(
    cellText=rows,
    colLabels=['Structure','N','a(Å)','b(Å)','c(Å)','α','V(Å³)','SA(m²/g)','Void'],
    loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 2.5)
for (r,c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#9FA8DA')
    if r == 0:
        cell.set_facecolor('#1A237E')
        cell.set_text_props(color='white', fontweight='bold')
    elif r == 1: cell.set_facecolor('#E3F2FD')
    elif r == 2: cell.set_facecolor('#FFEBEE')
    elif r == 3: cell.set_facecolor('#E8F5E9')
ax_cp.set_title('Full Property Summary', fontsize=10, fontweight='bold', pad=25)

plt.savefig('MOF_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Final printed summary ─────────────────────────────────────────────────────
print("=" * 65)
print("COMPLETE CHARACTERIZATION SUMMARY")
print("=" * 65)
for name in names:
    s = summaries[name]
    b = bet_results[name]
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    print(f"  Atoms          : {s['n_atoms']}")
    print(f"  Cell (a,b,c)   : {s['a']:.3f}  {s['b']:.3f}  {s['c']:.3f}  Å")
    print(f"  Angles (α,β,γ) : {s['alpha']:.1f}°  {s['beta']:.1f}°  {s['gamma']:.1f}°")
    print(f"  Volume         : {s['volume']:.1f} Å³")
    print(f"  Composition    : " +
          "  ".join(f"{e}: {n}" for e,n in sorted(s['el_count'].items())))
    print(f"  BET SA         : {b['sa_m2_per_g']:.0f} m²/g  ({b['sa_A2']:.0f} Å²)")
    print(f"  Void fraction  : {b['void_fraction']*100:.1f}%")
    print(f"  Pore volume    : {b['pore_volume_A3']:.0f} Å³")
    if s['bond_stats']:
        print(f"  Bond lengths:")
        for (e1,e2), bs in sorted(s['bond_stats'].items()):
            bar = '█' * int(bs['mean'] * 10)
            print(f"    {e1}–{e2}:  {bs['mean']:.3f} ± {bs['std']:.4f} Å  (N={bs['n']})  {bar}")
    if name in ftir_results:
        print(f"  FTIR modes:")
        for p in ftir_results[name][2]:
            print(f"    {p['wn']:6.0f} cm⁻¹  k={p['k_kcal']:5.1f} kcal/mol/Å²  [{p['label']}]")


---
## 8  Pore Size Distribution (PSD)

### Experimental technique

PSD is measured experimentally by applying the **BJH (Barrett-Joyner-Halenda)**
or **DFT/NLDFT** method to the N₂ adsorption isotherm — different branches of
the isotherm correspond to pores of different sizes filling and emptying.
For microporous MOFs (pores < 2 nm), NLDFT is preferred over BJH.

---

### What PSD shows

| Region | Pore type | Size |
|--------|-----------|------|
| Micropores | < 2 nm | Most MOFs — UiO-66 ~6–8 Å, MIL-101 ~14–34 Å |
| Mesopores | 2–50 nm | Large-pore MOFs, hierarchical structures |
| Macropores | > 50 nm | Rare in pristine MOFs |

The PSD tells you:
- **Where** molecules can sit (pore diameter > 2 × molecular radius)
- **How many** distinct pore environments exist (number of peaks)
- **Bottleneck diameter** — the narrowest passage a molecule must fit through to diffuse

---

### How we compute it geometrically

This is the **largest sphere algorithm** (equivalent to what Zeo++ does):

For each random test point **p** in the box:

1. Compute the distance from **p** to every atom center (with PBC minimum image)
2. Subtract the atom's vdW radius → this gives the "clearance" to each atom's surface
3. The largest sphere centered at **p** that fits without overlap has radius:

$$
r_{\text{probe}}(\mathbf{p}) = \min_j \left( d(\mathbf{p}, \mathbf{r}_j) - r_{\text{vdW},j} \right)
$$

4. If $r_{\text{probe}} > 0$, point **p** is in the void space

The **PSD** is the histogram of $r_{\text{probe}}$ values over all void-space points,
weighted by the shell volume $4\pi r^2$ (so it represents the volume contribution
of each pore size, not just the count of test points).

Two characteristic diameters are reported:
- **D_i** (largest included sphere) — largest sphere that fits anywhere in the pore
- **D_f** (largest free sphere) — largest sphere that can pass *through* the pore
  network (the bottleneck / limiting aperture)

> **Note:** For a truly accurate D_f you need Dijkstra path-finding on a grid,
> which requires Zeo++ or a full Voronoi decomposition. Here we approximate D_f
> as the diameter at the *minimum* of the PSD between two peaks (the saddle point),
> which works well for MOFs with distinct cage + window geometry like UiO-66 and MIL-101.

In [ ]:
def compute_psd(data, n_samples=200000, r_max_pore=20.0, dr=0.05):
    """
    Compute PSD geometrically via Monte Carlo sampling.

    For each random void-space point, compute:
        r_probe(p) = min_j( d(p, atom_j) - r_vdW_j )

    This is the radius of the largest sphere centered at p
    that fits without overlapping any atom.

    D_i = 2 * max(r_probe)  — largest included sphere diameter [reliable]
    D_f = NOT computed here — requires Zeo++ (Voronoi + Dijkstra path search)

    The volume-weighted PSD (hist * 4*pi*r^2) is reported because it weights
    pore environments by their volumetric contribution, matching what
    experiments like N2 adsorption preferentially sample.
    """
    atoms    = data['atoms']
    masses   = data['masses']
    cell     = data['cell']
    vol      = abs(np.linalg.det(cell))
    cell_inv = np.linalg.inv(cell.T)

    pos   = np.array([[a['x'], a['y'], a['z']] for a in atoms])
    els   = [element_from(masses[a['type']]['label'],
                          masses[a['type']]['mass']) for a in atoms]
    radii = np.array([VDW_RADII.get(e, 1.70) for e in els])

    np.random.seed(123)
    rand_cart = np.random.rand(n_samples, 3) @ cell.T

    r_probe_all = []
    block = 500
    for b in range(0, n_samples, block):
        pts  = rand_cart[b:b+block]
        dv   = pts[:, np.newaxis, :] - pos[np.newaxis, :, :]
        df   = dv @ cell_inv.T
        df  -= np.round(df)
        dist = np.linalg.norm(df @ cell.T, axis=2)
        clearance = dist - radii[np.newaxis, :]
        r_probe_all.append(clearance.min(axis=1))

    r_probe_all  = np.concatenate(r_probe_all)
    void_mask    = r_probe_all > 0
    r_void       = r_probe_all[void_mask]
    void_fraction = void_mask.mean()

    # D_i: diameter of the largest sphere that fits anywhere in the pores
    # = 2 * maximum r_probe found across all void-space points
    # This is exact and reliable.
    d_i = r_void.max() * 2

    # Build histogram
    r_bins    = np.arange(0, r_void.max() + dr, dr)
    r_centers = (r_bins[:-1] + r_bins[1:]) / 2
    hist, _   = np.histogram(r_void, bins=r_bins)

    # Count-weighted PSD: raw probability of finding a void point of size r
    psd_count = hist.astype(float)
    if psd_count.max() > 0:
        psd_count = psd_count / psd_count.max()

    # Volume-weighted PSD: each point weighted by 4*pi*r^2
    # Represents how much pore volume exists at each length scale
    psd_vol = hist * 4 * np.pi * r_centers**2
    if psd_vol.max() > 0:
        psd_vol = psd_vol / psd_vol.max()

    # Dominant pore diameter from volume-weighted peak
    psd_vol_sm = gaussian_filter1d(psd_vol, sigma=2)
    d_peak_vol = r_centers[np.argmax(psd_vol_sm)] * 2

    # CDF-based percentile diameters
    cdf = np.cumsum(hist).astype(float)
    cdf /= cdf.max()
    d_50 = r_centers[np.searchsorted(cdf, 0.50)] * 2
    d_90 = r_centers[np.searchsorted(cdf, 0.90)] * 2

    return {
        'r':            r_centers,
        'psd_vol':      psd_vol,
        'psd_count':    psd_count,
        'd_i':          d_i,
        'd_peak_vol':   d_peak_vol,   # volume-weighted peak diameter
        'd_50':         d_50,          # median pore diameter
        'd_90':         d_90,          # 90th percentile diameter
        'void_fraction': void_fraction,
    }


# ── Compute PSD for all structures ────────────────────────────────────────────
print("Computing PSD...")
psd_results = {}
for name, data in structures.items():
    res = compute_psd(data, n_samples=200000)
    psd_results[name] = res
    print(f"\n  {name}:")
    print(f"    D_i  (largest included sphere)  = {res['d_i']:.2f} Å  — exact")
    print(f"    D_pk (volume-PSD peak diameter) = {res['d_peak_vol']:.2f} Å  — dominant pore size")
    print(f"    D_50 (median pore diameter)     = {res['d_50']:.2f} Å")
    print(f"    D_90 (90th percentile)          = {res['d_90']:.2f} Å")
    print(f"    Void fraction                   = {res['void_fraction']*100:.1f}%")
    print(f"    ⚠ D_f (free sphere diameter) requires Zeo++ — not computed here")

In [ ]:
MOLECULES = {
    'H₂':     2.89, 'H₂O':    2.65, 'CO₂':    3.30,
    'O₂':     3.46, 'N₂':     3.64, 'CH₄':    3.80,
    'SF₆':    5.50, 'Benzene': 5.85,
}
MOL_COLORS = ['#1565C0','#6A1B9A','#B71C1C','#00838F',
               '#2E7D32','#E65100','#F9A825','#4E342E']

fig, axes = plt.subplots(3, 1, figsize=(13, 13), sharex=True)
fig.suptitle(
    'Pore Size Distribution  (PSD)\n'
    r'$r_\mathrm{probe}(\mathbf{p}) = \min_j\left(d(\mathbf{p},\mathbf{r}_j) - r_{\mathrm{vdW},j}\right)$'
    '   —   volume-weighted,  normalised',
    fontsize=12, fontweight='bold'
)

for ax, name in zip(axes, structures):
    res = psd_results[name]
    col = COLORS[name]
    r   = res['r']

    psd_vol_sm   = gaussian_filter1d(res['psd_vol'],   sigma=2)
    psd_count_sm = gaussian_filter1d(res['psd_count'], sigma=2)
    ymax = psd_vol_sm.max() if psd_vol_sm.max() > 0 else 1.0

    # ── PSD curves ────────────────────────────────────────────────────────────
    ax.plot(r*2, psd_vol_sm,   color=col, lw=2.2, label='Volume-weighted PSD')
    ax.plot(r*2, psd_count_sm, color=col, lw=1.0, ls='--', alpha=0.45,
            label='Count-weighted PSD')
    ax.fill_between(r*2, 0, psd_vol_sm, alpha=0.12, color=col)

    # ── D_i line ──────────────────────────────────────────────────────────────
    di   = res['d_i']
    dpk  = res['d_peak_vol']
    d50  = res['d_50']
    d90  = res['d_90']

    ax.axvline(di,  color=col, lw=1.8, ls='-.', alpha=0.9,
               label=f'$D_i$ = {di:.1f} Å  (largest included sphere — exact)')
    ax.axvline(dpk, color=col, lw=1.3, ls=':',  alpha=0.7,
               label=f'$D_{{peak}}$ = {dpk:.1f} Å  (volume-PSD peak)')

    # ── CDF shading ───────────────────────────────────────────────────────────
    ax.axvspan(0, d50, alpha=0.06, color=col)
    ax.text(d50*0.5, ymax*0.15, f'50%\n< {d50:.1f} Å',
            ha='center', fontsize=7, color=col, alpha=0.8)

    # ── Molecule kinetic diameters ────────────────────────────────────────────
    for (mol, kd), mcol in zip(MOLECULES.items(), MOL_COLORS):
        if kd < r[-1]*2:
            ax.axvline(kd, color=mcol, lw=0.9, ls=':', alpha=0.65)
            ax.text(kd+0.1, ymax*0.97, mol, fontsize=7,
                    color=mcol, va='top', rotation=90, alpha=0.85)

    # ── Labels ────────────────────────────────────────────────────────────────
    bold_labels(ax, ylabel='PSD  (normalised)')
    ax.set_ylim(0, ymax*1.35)
    ax.text(0.02, 0.93, name, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color=col)
    ax.text(0.98, 0.93,
            f'$D_i$ = {di:.1f} Å    $D_{{peak}}$ = {dpk:.1f} Å    '
            f'$D_{{50}}$ = {d50:.1f} Å    void = {res["void_fraction"]*100:.1f}%',
            transform=ax.transAxes, fontsize=8.5, ha='right', color=col, fontweight='bold')
    ax.legend(fontsize=8, loc='upper center', ncol=2)
    style_ax(ax)

bold_labels(axes[-1], xlabel='Pore diameter  (Å)')

# ── Note about D_f ────────────────────────────────────────────────────────────
fig.text(0.5, 0.01,
         '⚠  D_f (largest free sphere / bottleneck diameter) requires Voronoi decomposition '
         '+ Dijkstra path search (Zeo++). UiO-66 (MLIP) uses a small primitive cell, so its\n'
         'PSD is capped near half the cell length by periodic boundary conditions — not a true '
         'bulk pore-size limit. D_peak is the volume-weighted dominant pore size.',
         ha='center', fontsize=8.5, style='italic', color='#555',
         bbox=dict(boxstyle='round', facecolor='#fffde7', alpha=0.8))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('PSD.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Molecule accessibility table ──────────────────────────────────────────────
print(f"\n{'Molecule':<12} {'d_k (Å)':>8}", end='')
for name in structures:
    short = name.replace('UiO-66','UiO').replace('MIL-101','MIL')
    print(f"  {short:>14}", end='')
print(f"\n{'-'*70}")
for mol, kd in MOLECULES.items():
    print(f"{mol:<12} {kd:>8.2f}", end='')
    for name in structures:
        di = psd_results[name]['d_i']
        # A molecule fits in the cage if kd < D_i
        # We can't say if it diffuses through without D_f
        status = '✓ in cage' if kd < di else '✗ too large'
        print(f"  {status:>14}", end='')
    print()
print("\n⚠  'in cage' means molecule fits inside the largest pore.")
print("   Whether it can DIFFUSE through requires D_f from Zeo++.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# UiO-66 (UFF) — Full Characterization Figure (all 5 analyses)
# ══════════════════════════════════════════════════════════════════════════════

TARGET = 'UiO-66 (UFF)'
col    = COLORS[TARGET]

fig = plt.figure(figsize=(18, 14))
fig.suptitle(f'Full Characterization — {TARGET}',
             fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# ── Shared IR band definitions ────────────────────────────────────────────────
IR_BANDS = [
    (400,  700,  'M–O',    '#FFCDD2'),
    (700,  900,  'C–H\nbend', '#C8E6C9'),
    (1000, 1200, 'C–O',    '#BBDEFB'),
    (1350, 1650, 'COO⁻',   '#FFE0B2'),
    (2850, 3050, 'C–H\nstr.','#B2EBF2'),
    (3200, 3700, 'O–H',    '#FFF9C4'),
]

# ── Molecule kinetic diameters ────────────────────────────────────────────────
MOLECULES = {
    'H₂O': 2.65, 'H₂': 2.89, 'CO₂': 3.30,
    'O₂':  3.46, 'N₂': 3.64, 'CH₄': 3.80,
}
MOL_COLS = ['#6A1B9A','#1565C0','#B71C1C',
            '#00838F','#2E7D32','#E65100']

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 1 (top-left): XRD
# ─────────────────────────────────────────────────────────────────────────────
ax_xrd = fig.add_subplot(gs[0, 0])

tt, I = xrd_results[TARGET]
ax_xrd.plot(tt, I, color=col, lw=1.2)
ax_xrd.fill_between(tt, 0, I, alpha=0.12, color=col)

pks, _ = find_peaks(I, height=6, distance=12)
for pk in pks[np.argsort(I[pks])[::-1][:6]]:
    ax_xrd.annotate(f'{tt[pk]:.2f}°', xy=(tt[pk], I[pk]),
                    xytext=(0, 5), textcoords='offset points',
                    fontsize=7, ha='center', color=col,
                    arrowprops=dict(arrowstyle='-', lw=0.4, color=col))

lp = summaries[TARGET]
ax_xrd.text(0.98, 0.96,
            f"a = {lp['a']:.2f} Å\nV = {lp['volume']:.0f} Å³",
            transform=ax_xrd.transAxes, fontsize=8,
            ha='right', va='top', color='gray')
ax_xrd.set_xlabel('2θ (degrees)', fontsize=10)
ax_xrd.set_ylabel('Intensity (a.u.)', fontsize=10)
ax_xrd.set_title('Powder XRD  (Cu Kα, λ=1.5406 Å)', fontsize=10, fontweight='bold')
ax_xrd.set_xlim(2, 45)
ax_xrd.set_ylim(-2, 118)
ax_xrd.grid(True, alpha=0.2, lw=0.5)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 2 (top-middle): g(r)
# ─────────────────────────────────────────────────────────────────────────────
ax_gr = fig.add_subplot(gs[0, 1])

PAIR_COLS = {
    ('C','O'):'#FF6F00', ('O','O'):'#C62828', ('C','C'):'#6A1B9A',
    ('O','Zr'):'#0277BD', ('H','O'):'#78909C', ('H','C'):'#A1887F',
}

r, g, pairs, rho = gr_results[TARGET]
g_sm = gaussian_filter1d(g, sigma=1.2)
ax_gr.plot(r, g_sm, color=col, lw=2.0, label='Total', zorder=5)
ax_gr.axhline(1, color='k', lw=0.6, ls='--', alpha=0.35)

for key, ph in sorted(pairs.items()):
    pcol = PAIR_COLS.get(key, None)
    if pcol:
        ph_sm = gaussian_filter1d(ph, sigma=1.2)
        mx = ph_sm.max()
        if mx > 0:
            ax_gr.plot(r, ph_sm/mx * g_sm.max()*0.55,
                       color=pcol, lw=0.9, alpha=0.75, ls='--',
                       label=f'g({key[0]}–{key[1]})')

# Label first-shell peaks
r_pks, _ = find_peaks(g_sm, height=1.5, distance=8)
for pk in r_pks[:6]:
    ax_gr.text(r[pk]+0.07, g_sm[pk]*0.91, f'{r[pk]:.2f} Å',
               fontsize=7, color=col, fontweight='bold')

ax_gr.set_xlabel('r  (Å)', fontsize=10)
ax_gr.set_ylabel('g(r)', fontsize=10)
ax_gr.set_title('Radial Distribution Function', fontsize=10, fontweight='bold')
ax_gr.set_xlim(0.8, 10)
ax_gr.set_ylim(0, g_sm.max()*1.35)
ax_gr.legend(fontsize=7, ncol=2, loc='upper right', framealpha=0.7)
ax_gr.grid(True, alpha=0.2, lw=0.5)
ax_gr.text(0.02, 0.94, f'ρ = {rho:.5f} atoms/Å³',
           transform=ax_gr.transAxes, fontsize=7.5, color='gray')

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 3 (top-right): BET — both void fractions
# ─────────────────────────────────────────────────────────────────────────────
ax_bet = fig.add_subplot(gs[0, 2])

b = bet_results[TARGET]
categories  = ['SA\n(m²/g)', 'Geometric\nvoid (%)', 'N₂-accessible\nvoid (%)']
values      = [b['sa_m2_per_g'], b['void_geometric']*100, b['void_accessible']*100]
bar_colors  = ['#42A5F5', '#66BB6A', '#FFA726']

bars = ax_bet.bar(categories, values, color=bar_colors,
                  edgecolor='k', lw=0.7, alpha=0.88, width=0.55)
for bar, v in zip(bars, values):
    ax_bet.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + max(values)*0.02,
                f'{v:.0f}', ha='center', va='bottom',
                fontsize=11, fontweight='bold')

# Literature reference lines
ax_bet.axhline(1300, color='#42A5F5', lw=1.2, ls='--', alpha=0.6,
               label='Lit. SA ~1200–1400')
ax_bet.axhline(50,   color='#66BB6A', lw=1.2, ls='--', alpha=0.6,
               label='Lit. geom. void ~48–52%')

ax_bet.set_ylabel('Value', fontsize=10)
ax_bet.set_title('BET Surface Area & Void Fractions', fontsize=10, fontweight='bold')
ax_bet.set_ylim(0, max(values)*1.35)
ax_bet.legend(fontsize=7.5, loc='upper right', framealpha=0.7)
ax_bet.grid(True, axis='y', alpha=0.25, lw=0.5)

# Add probe radius note
ax_bet.text(0.02, 0.02,
            'Probe radius = 1.55 Å (N₂)',
            transform=ax_bet.transAxes, fontsize=7.5,
            color='gray', style='italic')

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 4 (bottom-left + bottom-middle): FTIR — wider panel
# ─────────────────────────────────────────────────────────────────────────────
ax_ir = fig.add_subplot(gs[1, 0:2])   # spans two columns

wn, spec, ir_peaks = ftir_results[TARGET]
spec_sm = gaussian_filter1d(spec, sigma=3)
trans   = 100.0 - spec_sm

for x0, x1, lbl, rcol in IR_BANDS:
    ax_ir.axvspan(x0, x1, alpha=0.15, color=rcol, zorder=0)
    ax_ir.text((x0+x1)/2, 102, lbl, ha='center', va='bottom',
               fontsize=8, color='#555')

ax_ir.plot(wn, trans, color=col, lw=1.8, zorder=5)
ax_ir.fill_between(wn, 100, trans, alpha=0.14, color=col)

for p in ir_peaks:
    ax_ir.axvline(p['wn'], color=col, lw=0.4, alpha=0.2, ls=':')

pidx, _ = find_peaks(spec_sm, height=12, distance=25)
for pk in pidx[np.argsort(spec_sm[pidx])[::-1][:8]]:
    ax_ir.annotate(f"{wn[pk]:.0f}",
                   xy=(wn[pk], trans[pk]),
                   xytext=(4, 6), textcoords='offset points',
                   fontsize=8.5, color=col, rotation=45, ha='left',
                   fontweight='bold')

ax_ir.set_xlim(4000, 400)
ax_ir.set_ylim(0, 105)
ax_ir.set_yticks([])
ax_ir.set_xlabel('Wavenumber  (cm⁻¹)', fontsize=10)
ax_ir.set_ylabel('Transmittance (a.u.)', fontsize=10)
ax_ir.set_title('Simulated FTIR  '
                r'$\tilde{\nu} = \frac{1}{2\pi c}\sqrt{\frac{k}{\mu}}$',
                fontsize=10, fontweight='bold')
ax_ir.grid(True, alpha=0.2, lw=0.5)
ax_ir.text(0.50, 0.12,
           '⚠  COO⁻ bands (~1390, ~1580 cm⁻¹) absent\n'
           '    — require full normal mode analysis',
           transform=ax_ir.transAxes, fontsize=8, ha='center',
           color='#B71C1C', style='italic',
           bbox=dict(boxstyle='round', facecolor='#FFEBEE',
                     alpha=0.85, edgecolor='#B71C1C'))

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 5 (bottom-right): PSD
# ─────────────────────────────────────────────────────────────────────────────
ax_psd = fig.add_subplot(gs[1, 2])

res     = psd_results[TARGET]
r_psd   = res['r']
psd_sm  = gaussian_filter1d(res['psd_vol'],   sigma=2)
psdc_sm = gaussian_filter1d(res['psd_count'], sigma=2)
ymax    = psd_sm.max() if psd_sm.max() > 0 else 1.0

ax_psd.plot(r_psd*2, psd_sm,  color=col, lw=2.0, label='Volume-weighted')
ax_psd.plot(r_psd*2, psdc_sm, color=col, lw=1.0, ls='--',
            alpha=0.45, label='Count-weighted')
ax_psd.fill_between(r_psd*2, 0, psd_sm, alpha=0.12, color=col)

# D_i line
di  = res['d_i']
dpk = res['d_peak_vol']
ax_psd.axvline(di,  color=col, lw=1.6, ls='-.',
               label=f'$D_i$ = {di:.1f} Å')
ax_psd.axvline(dpk, color=col, lw=1.2, ls=':',
               label=f'$D_{{peak}}$ = {dpk:.1f} Å')

# Molecule kinetic diameters
for (mol, kd), mcol in zip(MOLECULES.items(), MOL_COLS):
    if kd < r_psd[-1]*2:
        ax_psd.axvline(kd, color=mcol, lw=1.0, ls=':', alpha=0.75)
        ax_psd.text(kd+0.1, ymax*0.97, mol,
                    fontsize=7, color=mcol, va='top',
                    rotation=90, alpha=0.9)

ax_psd.set_xlabel('Pore diameter  (Å)', fontsize=10)
ax_psd.set_ylabel('PSD  (normalised)', fontsize=10)
ax_psd.set_title('Pore Size Distribution', fontsize=10, fontweight='bold')
ax_psd.set_xlim(0, 12)
ax_psd.set_ylim(0, ymax*1.35)
ax_psd.legend(fontsize=7.5, loc='upper right', framealpha=0.7)
ax_psd.grid(True, alpha=0.2, lw=0.5)
ax_psd.text(0.02, 0.02,
            '⚠ D_f needs Zeo++',
            transform=ax_psd.transAxes, fontsize=7.5,
            color='gray', style='italic')

# ─────────────────────────────────────────────────────────────────────────────
plt.savefig('UiO66_UFF_full_characterization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: UiO66_UFF_full_characterization.png")

---
## 6  Experimental Reference Extraction (UiO-66)  <a name="6-extraction"></a>

`UiO-66/Experiment for UiO.pptx` is a slide deck (a zip archive — pictures live
under `ppt/media/`) with many embedded images. Of those, three are usable
experimental reference plots for **pure UiO-66** (no Fe-loading):
- Powder XRD (image7.png)
- N₂ physisorption isotherm, used for BET (image5.png — also shows the
  Fe15@UiO-66 composite curve from an unrelated study; ignore that one)
- FTIR (image8.png)

The rest are university branding slides, an isosteric-enthalpy plot, a TGA
heat-capacity curve, instrument/rig photos, and thermal-camera stills — not
used here. Since UiO-66 (UFF) and UiO-66 (MLIP) are two computational
treatments of the *same real material*, these three experimental references
are reused as the "(a)" panel for **both** variants below.


In [ ]:
os.makedirs(EXPDIR['UiO-66 (UFF)'], exist_ok=True)
SRC_PPTX = os.path.join('UiO-66', 'Experiment for UiO.pptx')

with zipfile.ZipFile(SRC_PPTX) as z:
    media = sorted(n for n in z.namelist() if n.startswith('ppt/media/'))
    print(f'{len(media)} embedded images found in the .pptx:')
    for m in media:
        print(' ', m)

    # Manually identified by visual inspection (see markdown above):
    #   image7.png -> powder XRD, pure UiO-66 only
    #   image5.png -> N2 physisorption isotherm (UiO-66 + Fe15@UiO-66 composite)
    #   image8.png -> FTIR, pure UiO-66 only
    KEEP = {
        'ppt/media/image7.png': 'UiO66_exp_XRD.png',
        'ppt/media/image5.png': 'UiO66_exp_BET.png',
        'ppt/media/image8.png': 'UiO66_exp_FTIR.png',
    }
    for src, dst in KEEP.items():
        with z.open(src) as fin, open(os.path.join(EXPDIR['UiO-66 (UFF)'], dst), 'wb') as fout:
            shutil.copyfileobj(fin, fout)
        print(f'extracted {src} -> {os.path.join(EXPDIR["UiO-66 (UFF)"], dst)}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, fname, title in zip(
        axes,
        ['UiO66_exp_XRD.png', 'UiO66_exp_BET.png', 'UiO66_exp_FTIR.png'],
        ['Experimental XRD', 'Experimental N$_2$ isotherm (BET)', 'Experimental FTIR']):
    img = mpimg.imread(os.path.join(EXPDIR['UiO-66 (UFF)'], fname))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')
plt.suptitle('Extracted experimental references — pure UiO-66', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ── XRD: UiO-66 (UFF) — two-panel with experimental reference ─────────────────
TARGET = 'UiO-66 (UFF)'
col    = COLORS[TARGET]
lp     = summaries[TARGET]
tt, I  = xrd_results[TARGET]

def plot_xrd_uff(ax):
    ax.plot(tt, I, color=col, lw=1.2)
    ax.fill_between(tt, 0, I, alpha=0.12, color=col)
    pks, _ = find_peaks(I, height=6, distance=12)
    for pk in pks[np.argsort(I[pks])[::-1][:8]]:
        ax.annotate(f'{tt[pk]:.2f}°',
                    xy=(tt[pk], I[pk]),
                    xytext=(0, 7), textcoords='offset points',
                    fontsize=8, ha='center', color=col,
                    arrowprops=dict(arrowstyle='-', lw=0.5, color=col))
    bold_labels(ax, xlabel='2θ (degrees)', ylabel='Intensity (a.u.)')
    ax.set_xlim(2, 45)
    ax.set_ylim(-2, 118)
    ax.text(0.02, 0.95, TARGET, transform=ax.transAxes,
            fontsize=11, fontweight='bold', color=col)
    style_ax(ax)

fig, (ax_exp, ax_comp) = plt.subplots(1, 2, figsize=(15, 6))
img = mpimg.imread(os.path.join(EXPDIR[TARGET], 'UiO66_exp_XRD.png'))
ax_exp.imshow(img); ax_exp.axis('off')
ax_exp.set_title('(a) Experimental', fontsize=12, fontweight='bold', loc='left')
plot_xrd_uff(ax_comp)
ax_comp.set_title('(b) Computational', fontsize=12, fontweight='bold', loc='left')
fig.suptitle(f'Simulated Powder XRD — {TARGET}', fontsize=13, fontweight='bold')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_XRD.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
# ── g(r): UiO-66 (UFF) ────────────────────────────────────────────────────────
TARGET = 'UiO-66 (UFF)'
col    = COLORS[TARGET]

PAIR_COLS = {
    ('C','O'): '#FF6F00', ('O','O'): '#C62828', ('C','C'): '#6A1B9A',
    ('O','Zr'):'#0277BD', ('H','O'): '#78909C', ('H','C'): '#A1887F',
}

r, g, pairs, rho = gr_results[TARGET]
g_sm = gaussian_filter1d(g, sigma=1.2)

fig, ax = plt.subplots(1, 1, figsize=(11, 6))

ax.plot(r, g_sm, color=col, lw=2.2, label='Total g(r)', zorder=5)
ax.axhline(1, color='k', lw=0.7, ls='--', alpha=0.4, label='Ideal gas (g=1)')
ax.fill_between(r, 0, np.minimum(g_sm, 1), alpha=0.06, color=col)

for key, ph in sorted(pairs.items()):
    pcol = PAIR_COLS.get(key, None)
    if pcol:
        ph_sm = gaussian_filter1d(ph, sigma=1.2)
        mx = ph_sm.max()
        if mx > 0:
            ax.plot(r, ph_sm/mx * g_sm.max()*0.55,
                    color=pcol, lw=1.0, alpha=0.8, ls='--',
                    label=f'g({key[0]}–{key[1]})')

r_pks, _ = find_peaks(g_sm, height=1.5, distance=8)
for pk in r_pks[:6]:
    ax.text(r[pk]+0.07, g_sm[pk]*0.91, f'{r[pk]:.2f} Å',
            fontsize=8, color=col, fontweight='bold')

bold_labels(ax, xlabel='r  (Å)', ylabel='g(r)')
ax.set_title('Total & Partial g(r)', fontsize=11, fontweight='bold')
ax.set_xlim(0.8, 10)
ax.set_ylim(0, g_sm.max()*1.35)
ax.legend(fontsize=8, ncol=2, loc='upper right')
style_ax(ax)
ax.text(0.02, 0.95, f'ρ = {rho:.5f} atoms/Å³',
        transform=ax.transAxes, fontsize=8, color='gray')

fig.suptitle(f'Radial Distribution Function — {TARGET}',
             fontsize=12, fontweight='bold')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_gr_total_partial.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
# ── BET: UiO-66 (UFF) — two-panel with experimental reference ─────────────────
TARGET = 'UiO-66 (UFF)'
col    = COLORS[TARGET]
b      = bet_results[TARGET]

def plot_bet_uff(axes2):
    ax = axes2[0]
    bars = ax.bar([''], [b['sa_m2_per_g']], color='#42A5F5', edgecolor='k', lw=0.8,
                  alpha=0.85, width=0.45)
    ax.text(bars[0].get_x()+bars[0].get_width()/3, b['sa_m2_per_g'] + 30,
            f"{b['sa_m2_per_g']:.0f} m²/g", ha='center', fontsize=13,
            fontweight='bold', color='#1565C0')
    ax.axhspan(1200, 1600, alpha=0.12, color='green', label='Experimental range\n(1200–1600 m²/g)')
    ax.axhline(1200, color='green', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(1600, color='green', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(1200, color='navy', lw=1.2, ls=':',
               label='Ideal defect-free sim. (1200 m²/g)', alpha=0.7)
    bold_labels(ax, ylabel='Surface Area (m²/g)')
    ax.set_title('BET Surface Area', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 2000)
    ax.legend(fontsize=8.5, loc='upper right')
    style_ax(ax)

    ax = axes2[1]
    labels = ['Geometric void\n(probe = 0 Å)', 'N₂-accessible void\n(probe = 1.55 Å)']
    vals   = [b['void_geometric']*100, b['void_accessible']*100]
    bcols  = ['#66BB6A', '#FFA726']
    bars = ax.bar(labels, vals, color=bcols, edgecolor='k', lw=0.8, alpha=0.88, width=0.45)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v + 0.8, f'{v:.1f}%', ha='center',
                fontsize=13, fontweight='bold')
    ax.axhline(50, color='#388E3C', lw=1.3, ls='--', alpha=0.6, label='Literature geometric void')
    bold_labels(ax, ylabel='Void Fraction (%)')
    ax.set_title('Void Fraction Comparison', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 80)
    ax.legend(fontsize=8.5, loc='upper right')
    ax.set_xticklabels(ax.get_xticklabels(), fontweight='bold')
    style_ax(ax)

fig = plt.figure(figsize=(17, 5.5))
gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1.1, 1, 1])
ax_exp = fig.add_subplot(gs[0])
img = mpimg.imread(os.path.join(EXPDIR[TARGET], 'UiO66_exp_BET.png'))
ax_exp.imshow(img); ax_exp.axis('off')
ax_exp.set_title('(a) Experimental (N₂ isotherm)', fontsize=11, fontweight='bold', loc='left')
ax_b1 = fig.add_subplot(gs[1]); ax_b2 = fig.add_subplot(gs[2])
plot_bet_uff([ax_b1, ax_b2])
ax_b1.set_title('(b) ' + ax_b1.get_title(), fontsize=11, fontweight='bold')
fig.suptitle(f'BET Surface Area & Void Fractions — {TARGET}', fontsize=13, fontweight='bold')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_BET.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
# ── PSD: UiO-66 (UFF) — computation only ───────────────────────────────────────
TARGET = 'UiO-66 (UFF)'
col    = COLORS[TARGET]

res     = psd_results[TARGET]
r_psd   = res['r']
psd_sm  = gaussian_filter1d(res['psd_vol'],   sigma=2)
psdc_sm = gaussian_filter1d(res['psd_count'], sigma=2)
ymax    = psd_sm.max()
di      = res['d_i']
dpk     = res['d_peak_vol']
d50     = res['d_50']

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle(f'Pore Size Distribution — {TARGET}',
             fontsize=13, fontweight='bold')

ax.plot(r_psd*2, psd_sm,  color=col, lw=2.2, label='Volume-weighted')
ax.plot(r_psd*2, psdc_sm, color=col, lw=1.0, ls='--',
        alpha=0.45, label='Count-weighted')

ax.axvline(di,  color=col, lw=1.8, ls='-.', label=f'$D_i$ = {di:.1f} Å')
ax.axvline(dpk, color=col, lw=1.3, ls=':',  label=f'$D_{{peak}}$ = {dpk:.1f} Å')

ax.axvspan(0, d50, alpha=0.06, color=col)
ax.text(d50*0.5, ymax*0.12, f'50%\n< {d50:.1f} Å',
        ha='center', fontsize=8, color=col, alpha=0.8)

bold_labels(ax, xlabel='Pore diameter  (Å)', ylabel='PSD  (normalized)')
ax.set_xlim(0, 10)
ax.set_ylim(0, ymax*1.35)
ax.legend(fontsize=9, loc='upper center', ncol=2)
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_PSD.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
# ── FTIR: UiO-66 (UFF) — two-panel with experimental reference ────────────────
TARGET = 'UiO-66 (UFF)'
col    = COLORS[TARGET]

IR_BANDS = [
    (400,  700,  'M–O str.',  '#FFCDD2'),
    (700,  900,  'C–H bend',  '#C8E6C9'),
    (1000, 1200, 'C–O str.',  '#BBDEFB'),
    (1350, 1650, 'COO⁻',      '#FFE0B2'),
    (2850, 3050, 'C–H str.',  '#B2EBF2'),
    (3200, 3700, 'O–H str.',  '#FFF9C4'),
]

wn, spec, ir_peaks = ftir_results[TARGET]
spec_sm = gaussian_filter1d(spec, sigma=3)
trans   = 100.0 - spec_sm

def plot_ftir_uff(ax):
    for x0, x1, lbl, rcol in IR_BANDS:
        ax.axvspan(x0, x1, alpha=0.15, color=rcol, zorder=0)
        ax.text((x0+x1)/2, 102, lbl, ha='center', va='bottom',
                fontsize=8.5, color='#555')
    ax.plot(wn, trans, color=col, lw=1.8, zorder=5)
    ax.fill_between(wn, 100, trans, alpha=0.13, color=col)
    for p in ir_peaks:
        ax.axvline(p['wn'], color=col, lw=0.4, alpha=0.2, ls=':')
    pidx, _ = find_peaks(spec_sm, height=12, distance=25)
    for pk in pidx[np.argsort(spec_sm[pidx])[::-1][:8]]:
        ax.annotate(f"{wn[pk]:.0f}",
                    xy=(wn[pk], trans[pk]),
                    xytext=(4, 6), textcoords='offset points',
                    fontsize=9, color=col, rotation=45,
                    ha='left', fontweight='bold')
    bold_labels(ax, xlabel='Wavenumber  (cm⁻¹)', ylabel='Transmittance (a.u.)')
    ax.set_xlim(4000, 400)
    ax.set_ylim(0, 105)
    ax.set_yticks([])
    style_ax(ax)

fig, (ax_exp, ax_comp) = plt.subplots(1, 2, figsize=(15, 6))
img = mpimg.imread(os.path.join(EXPDIR[TARGET], 'UiO66_exp_FTIR.png'))
ax_exp.imshow(img); ax_exp.axis('off')
ax_exp.set_title('(a) Experimental', fontsize=12, fontweight='bold', loc='left')
plot_ftir_uff(ax_comp)
ax_comp.set_title('(b) Computational', fontsize=12, fontweight='bold', loc='left')
fig.suptitle(f'Simulated FTIR — {TARGET}', fontsize=13, fontweight='bold')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_FTIR.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONNECTIVITY FROM RDF — UiO-66 (UFF)
# Coordination number: N_AB = average number of B neighbours per A atom
#                    = (total A→B pairs within r_min) / N_A
#
# Correct expected values (averaged over ALL atoms of type A):
#   N(C–H)   = 0.50  → 4 of 8 C per BDC have 1 H; other 4 have none
#   N(C–O)   = 0.50  → 2 of 8 C per BDC have 2 O; other 6 have none
#   N(C–C)   = 2.00  → weighted avg: aromatic(2) + ipso(3) + carboxylate(1)
#   N(O–Zr)  = 1.50  → µ₃-O/OH: 3 Zr each; carboxylate O: 1 Zr each
#   N(Zr–O)  = 8.00  → each Zr: 4 µ₃-O/OH + 4 carboxylate O
#   N(Zr–Zr) = 4.00  → each Zr in Zr₆ octahedron has 4 nearest Zr
# ══════════════════════════════════════════════════════════════════════════════

TARGET = 'UiO-66 (UFF)'
data   = structures[TARGET]

def compute_coordination(data, pairs, r_max=5.0, dr=0.02):
    """
    Compute directed partial g_AB(r) and coordination numbers.

    Method
    ------
    1. For each ordered pair (A, B), build histogram of A→B distances
       using ALL atoms and PBC minimum image convention.
    2. Normalise to g_AB(r) = hist / (n_A * rho_B * 4*pi*r^2 * dr)
    3. Find first peak and first minimum r_min of g_AB(r).
    4. N_AB = sum(hist[r <= r_min]) / n_A
            = average number of B atoms within r_min of each A atom.
       This formula is exact — it does not depend on rho_B or shell volume.

    Parameters
    ----------
    pairs  : list of (el_A, el_B) tuples  — directed pairs to compute
    r_max  : float  maximum distance [Å]
    dr     : float  bin width [Å]
    """
    atoms    = data['atoms']
    masses   = data['masses']
    cell     = data['cell']
    vol      = abs(np.linalg.det(cell))
    cell_inv = np.linalg.inv(cell.T)

    pos = np.array([[a['x'], a['y'], a['z']] for a in atoms])
    els = [element_from(masses[a['type']]['label'],
                        masses[a['type']]['mass']) for a in atoms]

    el_count  = {e: els.count(e) for e in set(els)}
    pos_by_el = {e: pos[np.array([i for i,x in enumerate(els) if x==e])]
                 for e in set(els)}

    r_bins    = np.arange(0, r_max + dr, dr)
    r_centers = (r_bins[:-1] + r_bins[1:]) / 2
    results   = {}

    for el_A, el_B in pairs:
        pos_A = pos_by_el[el_A]
        pos_B = pos_by_el[el_B]
        n_A   = len(pos_A)
        n_B   = len(pos_B)
        hist  = np.zeros(len(r_centers))

        # Vectorised block loop over A atoms
        block = 100
        for b in range(0, n_A, block):
            chunk = pos_A[b:b+block]                         # (B, 3)
            dv    = chunk[:, np.newaxis, :] - pos_B[np.newaxis, :, :]   # (B, nB, 3)
            df    = dv @ cell_inv.T
            df   -= np.round(df)                             # minimum image
            d     = np.linalg.norm(df @ cell.T, axis=2)     # (B, nB)

            if el_A == el_B:
                # Exclude self-interaction (block offset index)
                for li in range(len(chunk)):
                    gi    = b + li                           # global index in pos_A
                    row   = d[li].copy()
                    row[gi] = 1e9                            # mask self
                    mask  = row < r_max
                    bidx  = np.floor(row[mask] / dr).astype(int)
                    np.add.at(hist, np.clip(bidx, 0, len(hist)-1), 1)
            else:
                mask = d < r_max
                bidx = np.floor(d[mask] / dr).astype(int)
                np.add.at(hist, np.clip(bidx, 0, len(hist)-1), 1)

        # Normalise to g(r) — for plotting and r_min detection
        rho_B = n_B / vol
        shell = 4 * np.pi * r_centers**2 * dr
        g_AB  = hist / (n_A * rho_B * shell + 1e-20)
        g_sm  = gaussian_filter1d(g_AB, sigma=3)

        # First peak
        pks, _ = find_peaks(g_sm, height=0.5, distance=5)
        if len(pks) == 0:
            continue
        fp    = pks[0]
        r_pk  = r_centers[fp]

        # First minimum after first peak
        se    = min(len(g_sm)-1, int(r_pk * 3.5 / dr))
        mi    = fp + np.argmin(g_sm[fp:se])
        r_mn  = r_centers[mi]

        # Coordination number — direct count, no normalisation ambiguity
        # N_AB = (all A→B pairs within r_min) / n_A
        N_AB  = hist[r_centers <= r_mn].sum() / n_A

        results[(el_A, el_B)] = {
            'r_peak': r_pk, 'r_min': r_mn,
            'N_AB':   N_AB,
            'g_AB':   g_sm,
            'hist':   hist,
        }
        print(f"  {el_A}-{el_B}: done")

    return results, r_centers, el_count


# ── Pairs to compute (directed: A is reference, B is neighbour) ───────────────
PAIRS = [
    ('C', 'H'),    # aromatic C–H bond
    ('C', 'O'),    # carboxylate C–O bond
    ('C', 'C'),    # aromatic/linker C–C bonds
    ('O', 'Zr'),   # O coordination around Zr
    ('Zr','O'),    # Zr coordination (should give 8)
    ('Zr','Zr'),   # Zr–Zr within Zr₆ cluster
    ('O', 'O'),    # O–O within Zr₆ cluster faces
]

# Correct expected values (per-atom averages, analytically derived)
EXPECTED = {
    ('C','H'):   ('0.50', 'half C atoms have 1 H (aromatic); other half have none'),
    ('C','O'):   ('0.50', 'only carboxylate C (2/8 per BDC) have 2 O'),
    ('C','C'):   ('2.00', 'aromatic:2 + ipso:3 + carboxylate:1 → avg 2.0'),
    ('O','Zr'):  ('1.50', 'µ₃-O/OH: 3 Zr; carboxylate-O: 1 Zr → avg 1.5'),
    ('Zr','O'):  ('8.00', '4 µ₃-O/OH + 4 carboxylate O per Zr'),
    ('Zr','Zr'): ('4.00', 'each Zr in Zr₆ octahedron has 4 Zr neighbours'),
    ('O','O'):   ('—',    'O–O within Zr₆ triangular face'),
}

print(f"Computing coordination numbers for {TARGET}...")
print("(Using all atoms — this takes ~2–3 minutes)")
coord, r_ctr, el_count = compute_coordination(data, PAIRS)

# ── Print table ───────────────────────────────────────────────────────────────
print(f"\n{'Pair':<10} {'r_peak (Å)':>11} {'r_min (Å)':>11} "
      f"{'N_AB':>8}   {'Expected':>8}   Physical meaning")
print('─' * 90)
for (A, B), res in sorted(coord.items(), key=lambda x: x[1]['r_peak']):
    exp_val, meaning = EXPECTED.get((A,B), ('—', ''))
    print(f"{A}-{B:<6}  {res['r_peak']:>11.3f} {res['r_min']:>11.3f} "
          f"{res['N_AB']:>8.2f}   {exp_val:>8}   {meaning}")

# ── Validation summary ────────────────────────────────────────────────────────
print(f"\nValidation:")
n_Zr = el_count['Zr']; n_O = el_count['O']
n_C  = el_count['C'];  n_H = el_count['H']
print(f"  N_atoms: C={n_C}, H={n_H}, O={n_O}, Zr={n_Zr}")
print(f"  BDC linkers = {n_C//8},  Zr₆ clusters = {n_Zr//6}")
print(f"  O types: µ₃-O={n_Zr//6*4}, µ₃-OH={n_Zr//6*4}, "
      f"carboxylate-O={n_C//8*4}")
print(f"  Zr-O bonds check: {n_Zr}×8 = {n_Zr*8}  vs  "
      f"O-side: {n_Zr//6*4*3}+{n_Zr//6*4*3}+{n_C//8*4*1} = "
      f"{n_Zr//6*4*3+n_Zr//6*4*3+n_C//8*4*1}")

# ── Plot ──────────────────────────────────────────────────────────────────────
PLOT_PAIRS = [('C','H'), ('C','O'), ('C','C'),
              ('O','Zr'), ('Zr','O'), ('Zr','Zr')]
COLS = {'C-H':'#2E7D32','C-O':'#FF6F00','C-C':'#6A1B9A',
        'O-Zr':'#0277BD','Zr-O':'#01579B','Zr-Zr':'#B71C1C'}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f'Partial g(r) and Coordination Numbers',
             fontsize=12, fontweight='bold')

for ax, (el_A, el_B) in zip(axes.flat, PLOT_PAIRS):
    if (el_A, el_B) not in coord:
        ax.set_visible(False); continue

    res = coord[(el_A, el_B)]
    col = COLS.get(f'{el_A}-{el_B}', '#1565C0')
    exp_val, _ = EXPECTED.get((el_A,el_B), ('—',''))

    ax.plot(r_ctr, res['g_AB'], color=col, lw=2.0)
    ax.axhline(1, color='k', lw=0.6, ls='--', alpha=0.35, label='g=1')
    ax.axvline(res['r_peak'], color=col, lw=1.0, ls=':', alpha=0.7,
               label=f"peak: {res['r_peak']:.3f} Å")
    ax.axvline(res['r_min'],  color=col, lw=1.2, ls='-.', alpha=0.7,
               label=f"$r_{{\\min}}$: {res['r_min']:.3f} Å")

    # Shade integration region
    mask = r_ctr <= res['r_min']
    ax.fill_between(r_ctr[mask], 0, res['g_AB'][mask],
                    alpha=0.18, color=col)

    bold_labels(ax, xlabel='r  (Å)', ylabel=f'g({el_A}–{el_B})')
    # ax.set_title(f'g({el_A}–{el_B})   '
    #              f'$N$ = {res["N_AB"]:.2f}  (expected {exp_val})',
    #              fontsize=10, fontweight='bold', color=col)
    ax.set_xlim(0.5, 5.0)
    ax.set_ylim(0, res['g_AB'][:int(5/0.02)].max() * 1.35)
    ax.legend(fontsize=8)
    style_ax(ax)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_coordination.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", os.path.join(OUTDIRS[TARGET], 'UiO66_UFF_coordination.png'))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MICROPORE VOLUME (V_mic) — UiO-66 (UFF)
#
# Method: geometric void fraction × cell volume / cell mass
#
#   V_mic [cm³/g] = phi_geom × V_cell [cm³] / M_cell [g]
#
# phi_geom is the geometric void fraction (probe = 0 Å),
# meaning the fraction of the simulation box NOT occupied by
# any atom's van der Waals sphere.
#
# Since UiO-66 is purely microporous (all pores < 2 nm as confirmed
# by the NLDFT PSD), V_meso ≈ 0 and V_mic ≈ V_total_pore.
#
# Literature experimental V_mic for UiO-66: 0.40–0.55 cm³/g
# (from N₂ adsorption at 77 K, Shearer et al. 2016)
# ══════════════════════════════════════════════════════════════════════════════

TARGET = 'UiO-66 (UFF)'
data   = structures[TARGET]

atoms    = data['atoms']
masses   = data['masses']
cell     = data['cell']
vol_A3   = abs(np.linalg.det(cell))          # cell volume [Å³]
cell_inv = np.linalg.inv(cell.T)

pos  = np.array([[a['x'], a['y'], a['z']] for a in atoms])
els  = [element_from(masses[a['type']]['label'],
                     masses[a['type']]['mass']) for a in atoms]

# vdW radii only — no probe (geometric void)
radii_bare = np.array([VDW_RADII.get(e, 1.70) for e in els])

# ── Monte Carlo integration of geometric void fraction ────────────────────────
N_mc = 200000
np.random.seed(42)
rand_pts = np.random.rand(N_mc, 3) @ cell.T   # random Cartesian points

n_void = 0
block  = 500
for b in range(0, N_mc, block):
    pts  = rand_pts[b:b+block]
    dv   = pts[:, np.newaxis, :] - pos[np.newaxis, :, :]
    df   = dv @ cell_inv.T
    df  -= np.round(df)                        # minimum image PBC
    dist = np.linalg.norm(df @ cell.T, axis=2)
    # A point is void if it does NOT overlap any atom's bare vdW sphere
    n_void += int(np.sum(~np.any(dist < radii_bare[np.newaxis, :], axis=1)))

phi_geom = n_void / N_mc                       # geometric void fraction [-]

# ── Convert to cm³/g ──────────────────────────────────────────────────────────
# Cell mass [g] = sum of atomic masses [amu] × amu_to_g
amu_to_g     = 1.6605e-24                      # 1 amu = 1.6605e-24 g
M_cell_g     = sum(masses[a['type']]['mass']
                   for a in atoms) * amu_to_g  # [g]

# Cell volume [cm³] (1 Å³ = 1e-24 cm³)
V_cell_cm3   = vol_A3 * 1e-24                  # [cm³]

# Micropore volume [cm³/g]
V_mic        = phi_geom * V_cell_cm3 / M_cell_g

# ── Also compute N₂-accessible pore volume for comparison ────────────────────
# Uses probe_r = 1.55 Å (N₂ kinetic radius) — closer to what experiment measures
probe_r      = 1.55
radii_probe  = radii_bare + probe_r

n_acc = 0
for b in range(0, N_mc, block):
    pts  = rand_pts[b:b+block]
    dv   = pts[:, np.newaxis, :] - pos[np.newaxis, :, :]
    df   = dv @ cell_inv.T;  df -= np.round(df)
    dist = np.linalg.norm(df @ cell.T, axis=2)
    n_acc += int(np.sum(~np.any(dist < radii_probe[np.newaxis, :], axis=1)))

phi_acc  = n_acc / N_mc
V_mic_N2 = phi_acc * V_cell_cm3 / M_cell_g

# ── Print results ─────────────────────────────────────────────────────────────
print(f"UiO-66 (UFF) — Micropore Volume")
print(f"{'─'*45}")
print(f"  Cell volume          : {vol_A3:.1f} Å³  =  {V_cell_cm3:.4e} cm³")
print(f"  Cell mass            : {M_cell_g:.4e} g")
print(f"  Geometric void frac. : {phi_geom*100:.1f}%  (probe = 0 Å)")
print(f"  N₂-accessible frac.  : {phi_acc*100:.1f}%  (probe = 1.55 Å)")
print(f"")
print(f"  V_mic (geometric)    : {V_mic:.4f} cm³/g")
print(f"  V_mic (N₂-probe)     : {V_mic_N2:.4f} cm³/g")
print(f"")
print(f"  Literature V_mic     : 0.40–0.55 cm³/g  (N₂ adsorption, 77 K)")
print(f"  Reference            : Shearer et al., Chem. Mater. 2016")
print(f"")
print(f"  Note: V_meso ≈ 0 (confirmed by NLDFT PSD — no pores above 2 nm)")
print(f"  → V_total_pore ≈ V_mic")

In [ ]:
# ── Total g(r) + 6 partial pairs (from coord, cell 32) — wide figure ─────────
# Requires: coord, r_ctr  (computed by cell 32 — run that first)

TARGET = 'UiO-66 (UFF)'
_col   = COLORS[TARGET]
r, g, pairs, rho = gr_results[TARGET]
g_sm = gaussian_filter1d(g, sigma=2.5)

# Pairs and colours matching cell 32
PLOT_PAIRS  = [('C','H'), ('C','O'), ('C','C'),
               ('O','Zr'), ('Zr','O'), ('Zr','Zr')]
PAIR_COLS32 = {
    ('C', 'H') : '#2E7D32',
    ('C', 'O') : '#FF6F00',
    ('C', 'C') : '#6A1B9A',
    ('O', 'Zr'): '#0277BD',
    ('Zr','O') : '#01579B',
    ('Zr','Zr'): '#B71C1C',
}
PAIR_LABELS32 = {
    ('C', 'H') : 'g(C–H)',
    ('C', 'O') : 'g(C–O)',
    ('C', 'C') : 'g(C–C)',
    ('O', 'Zr'): 'g(O–Zr)',
    ('Zr','O') : 'g(Zr–O)',
    ('Zr','Zr'): 'g(Zr–Zr)',
}

def _style_ax(ax, fs=12):
    ax.grid(False)
    for sp in ax.spines.values():
        sp.set_linewidth(2.2)
    ax.tick_params(width=1.5, length=5)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
        lbl.set_fontsize(fs - 1)

# ── Wide canvas ───────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 17))
gs_outer = gridspec.GridSpec(2, 1, figure=fig,
                              height_ratios=[1, 2],
                              hspace=0.32)

# ── TOP 1/3: Total g(r) ───────────────────────────────────────────────────────
ax_top = fig.add_subplot(gs_outer[0])
ax_top.plot(r, g_sm, color=_col, lw=2.6)
ax_top.fill_between(r, 0, g_sm, alpha=0.13, color=_col)
ax_top.set_xlabel('r  (Å)', fontweight='bold', fontsize=14)
ax_top.set_ylabel('g(r)',   fontweight='bold', fontsize=14)
ax_top.set_xlim(0.8, 12)
ax_top.set_ylim(0, g_sm.max() * 1.22)
_style_ax(ax_top, fs=13)

# ── BOTTOM 2/3: 2×3 properly-normalised partial g(r) from coord ──────────────
gs_inner = gridspec.GridSpecFromSubplotSpec(
    2, 3, subplot_spec=gs_outer[1],
    hspace=0.50, wspace=0.38
)

X_MAX = 5.0          # match cell 32 trim
dr32  = r_ctr[1] - r_ctr[0]
i_max = int(X_MAX / dr32)

for _idx, key in enumerate(PLOT_PAIRS):
    _row, _c = divmod(_idx, 3)
    ax = fig.add_subplot(gs_inner[_row, _c])

    if key in coord:
        g_ab = coord[key]['g_AB']       # already smoothed + normalised in cell 32
        y_max = g_ab[:i_max].max() * 1.30
        ax.plot(r_ctr, g_ab, color=PAIR_COLS32[key], lw=2.2)
        ax.set_ylim(0, y_max)

    ax.set_xlabel('r  (Å)',              fontweight='bold', fontsize=12)
    ax.set_ylabel(PAIR_LABELS32[key],    fontweight='bold', fontsize=12)
    ax.set_xlim(0.5, X_MAX)
    _style_ax(ax, fs=11)

plt.savefig(os.path.join(OUTDIRS['UiO-66 (UFF)'], 'UiO66_UFF_RDF.png'), dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', os.path.join(OUTDIRS['UiO-66 (UFF)'], 'UiO66_UFF_RDF.png'))


---
## 9  UiO-66 (MLIP) — Individual Characterization  <a name="9-mlip"></a>

Same physical material as above, computed from the bond-free MLIP primitive
rhombohedral cell (114 atoms, a=b=c≈14.77 Å, α=β=γ=60°) instead of the bonded
UFF 2×2×2 supercell. XRD / BET / PSD / RDF reuse the results already computed
in the shared section above (`xrd_results`, `bet_results`, `psd_results`,
`gr_results` all have a `'UiO-66 (MLIP)'` entry).

**FTIR** — this file has no `Bond Coeffs`, so the bond-frequency method
above doesn't apply. See the FTIR subsection right after this cell: forces
from the MACE and MTP potentials (both now dropped into `UiO-66/MLIP/`) feed
a phonopy finite-displacement phonon DOS instead.

**PSD caveat** — the primitive cell is small, so the largest-sphere search is
capped near half the cell length (~7 Å) by periodic boundary conditions; this
undersells the true pore size relative to the UFF supercell result.


### FTIR from the MACE potential

The bond-oscillator method above needs `Bond Coeffs`, which the MACE model
doesn't have — getting real frequencies out means actually evaluating forces
with it. This panel now reuses the finite-displacement phonon force constants
already computed (full precision, float64/CPU) by Test 2 of
`MACE_UiO66_N2_Reliability_Validation.ipynb` — no separate MACE relaxation is
done here. `mace_uio66_ftir.py` loads that Test 2 output
(`T2_phonopy_params_1x1x1.yaml`), runs a phonon density of states, and
converts it into an FTIR-style transmittance spectrum with the same
`phonon_dos_to_ftir()` method used for the UFF panels (peak *positions* are
the meaningful output — a phonon DOS has no dipole-derivative weighting, so
relative intensities are a rough proxy, not real IR intensities). It saves
`UiO66_MACE_FTIR.png`, which is just displayed below.

(An earlier version of this panel read a cached `mace_phonon_dos.npz` from a
different, older MACE model via `compute_mace_ftir.py`; that pipeline showed
no modes above 1600 cm⁻¹ and is no longer used.)


In [ ]:
# ── FTIR: UiO-66 (MACE), regenerated from Test 2's phonon force constants ─────
# Regenerate after any Test 2 rerun with:
#   cd .../UiO-66/MACE
#   OMP_NUM_THREADS=1 /Users/shayanjalalmanesh/anaconda3/envs/mace_val/bin/python \
#       mace_uio66_ftir.py mlip_validation_outputs/T2_phonopy_params_1x1x1.yaml
MACE_FTIR_PNG = os.path.join(OUTDIRS['UiO-66 (MLIP)'], 'UiO66_MACE_FTIR.png')
print("UiO-66 (MACE) FTIR figure present:", os.path.exists(MACE_FTIR_PNG), "->", MACE_FTIR_PNG)


In [ ]:
TARGET = 'UiO-66 (MACE)'

fig, (ax_exp, ax_comp) = plt.subplots(1, 2, figsize=(15, 6))
img_exp = mpimg.imread(os.path.join(EXPDIR['UiO-66 (MLIP)'], 'UiO66_exp_FTIR.png'))
ax_exp.imshow(img_exp); ax_exp.axis('off')
img_mace = mpimg.imread(MACE_FTIR_PNG)
ax_comp.imshow(img_mace); ax_comp.axis('off')

plt.tight_layout()
savepath = os.path.join(OUTDIRS['UiO-66 (MLIP)'], 'UiO66_MACE_FTIR_compare.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'UiO-66 (MLIP)'
col    = COLORS[TARGET]
tt, I  = xrd_results[TARGET]

def plot_xrd_mlip(ax):
    ax.plot(tt, I, color=col, lw=1.2)
    ax.fill_between(tt, 0, I, alpha=0.12, color=col)
    pks, _ = find_peaks(I, height=6, distance=8)
    for pk in pks[np.argsort(I[pks])[::-1][:8]]:
        ax.annotate(f'{tt[pk]:.2f}°', xy=(tt[pk], I[pk]),
                    xytext=(0, 7), textcoords='offset points',
                    fontsize=8, ha='center', color=col,
                    arrowprops=dict(arrowstyle='-', lw=0.5, color=col))
    bold_labels(ax, xlabel='2θ (degrees)', ylabel='Intensity (a.u.)')
    ax.set_xlim(2, 45)
    ax.set_ylim(-2, max(I.max()*1.25, 20))
    ax.text(0.02, 0.95, TARGET, transform=ax.transAxes,
            fontsize=11, fontweight='bold', color=col)
    style_ax(ax)

fig, (ax_exp, ax_comp) = plt.subplots(1, 2, figsize=(15, 6))
img = mpimg.imread(os.path.join(EXPDIR[TARGET], 'UiO66_exp_XRD.png'))
ax_exp.imshow(img); ax_exp.axis('off')
ax_exp.set_title('(a) Experimental', fontsize=12, fontweight='bold', loc='left')
plot_xrd_mlip(ax_comp)
ax_comp.set_title('(b) Computational', fontsize=12, fontweight='bold', loc='left')
fig.suptitle(f'Simulated Powder XRD — {TARGET}', fontsize=13, fontweight='bold')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_MLIP_XRD.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'UiO-66 (MLIP)'
col    = COLORS[TARGET]
b      = bet_results[TARGET]

def plot_bet_mlip(axes2):
    ax = axes2[0]
    bars = ax.bar([''], [b['sa_m2_per_g']], color='#66BB6A', edgecolor='k', lw=0.8,
                  alpha=0.85, width=0.45)
    ax.text(bars[0].get_x()+bars[0].get_width()/3, b['sa_m2_per_g'] + 30,
            f"{b['sa_m2_per_g']:.0f} m²/g", ha='center', fontsize=13,
            fontweight='bold', color=col)
    ax.axhspan(1200, 1600, alpha=0.12, color='green', label='Experimental range\n(1200–1600 m²/g)')
    ax.axhline(1200, color='green', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(1600, color='green', lw=1.0, ls='--', alpha=0.5)
    bold_labels(ax, ylabel='Surface Area (m²/g)')
    ax.set_title('BET Surface Area', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 2000)
    ax.legend(fontsize=8.5, loc='upper right')
    style_ax(ax)

    ax = axes2[1]
    labels = ['Geometric void\n(probe = 0 Å)', 'N₂-accessible void\n(probe = 1.55 Å)']
    vals   = [b['void_geometric']*100, b['void_accessible']*100]
    bcols  = ['#66BB6A', '#FFA726']
    bars = ax.bar(labels, vals, color=bcols, edgecolor='k', lw=0.8, alpha=0.88, width=0.45)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v + 0.8, f'{v:.1f}%', ha='center',
                fontsize=13, fontweight='bold')
    ax.axhline(50, color='#388E3C', lw=1.3, ls='--', alpha=0.6, label='Literature geometric void')
    bold_labels(ax, ylabel='Void Fraction (%)')
    ax.set_title('Void Fraction Comparison', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 80)
    ax.legend(fontsize=8.5, loc='upper right')
    ax.set_xticklabels(ax.get_xticklabels(), fontweight='bold')
    style_ax(ax)

fig = plt.figure(figsize=(17, 5.5))
gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1.1, 1, 1])
ax_exp = fig.add_subplot(gs[0])
img = mpimg.imread(os.path.join(EXPDIR[TARGET], 'UiO66_exp_BET.png'))
ax_exp.imshow(img); ax_exp.axis('off')
ax_exp.set_title('(a) Experimental (N₂ isotherm)', fontsize=11, fontweight='bold', loc='left')
ax_b1 = fig.add_subplot(gs[1]); ax_b2 = fig.add_subplot(gs[2])
plot_bet_mlip([ax_b1, ax_b2])
ax_b1.set_title('(b) ' + ax_b1.get_title(), fontsize=11, fontweight='bold')
fig.suptitle(f'BET Surface Area & Void Fractions — {TARGET}', fontsize=13, fontweight='bold')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_MLIP_BET.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'UiO-66 (MLIP)'
col    = COLORS[TARGET]
res    = psd_results[TARGET]
r_psd   = res['r']
psd_sm  = gaussian_filter1d(res['psd_vol'],   sigma=2)
psdc_sm = gaussian_filter1d(res['psd_count'], sigma=2)
ymax    = psd_sm.max() if psd_sm.max() > 0 else 1.0
di, dpk, d50 = res['d_i'], res['d_peak_vol'], res['d_50']

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle(f'Pore Size Distribution — {TARGET}', fontsize=13, fontweight='bold')

ax.plot(r_psd*2, psd_sm,  color=col, lw=2.2, label='Volume-weighted')
ax.plot(r_psd*2, psdc_sm, color=col, lw=1.0, ls='--', alpha=0.45, label='Count-weighted')
ax.axvline(di,  color=col, lw=1.8, ls='-.', label=f'$D_i$ = {di:.1f} Å')
ax.axvline(dpk, color=col, lw=1.3, ls=':',  label=f'$D_{{peak}}$ = {dpk:.1f} Å')
ax.axvspan(0, d50, alpha=0.06, color=col)
ax.text(d50*0.5, ymax*0.12, f'50%\n< {d50:.1f} Å', ha='center', fontsize=8, color=col, alpha=0.8)

bold_labels(ax, xlabel='Pore diameter  (Å)', ylabel='PSD  (normalized)')
ax.set_xlim(0, 8)
ax.set_ylim(0, ymax*1.35)
ax.legend(fontsize=9, loc='upper center', ncol=2)
style_ax(ax)
ax.text(0.02, 0.02, 'Small primitive cell — PBC-capped near half the cell length',
        transform=ax.transAxes, fontsize=7.5, color='gray', style='italic')

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'UiO66_MLIP_PSD.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
# RDF for UiO-66 (MLIP): total g(r) (already computed above) + same 6 directed
# partials as the UFF figure (cell 33), computed fresh on the MLIP structure
# using compute_coordination (defined in the connectivity section above).
TARGET_MLIP = 'UiO-66 (MLIP)'
data_mlip   = structures[TARGET_MLIP]
col_mlip    = COLORS[TARGET_MLIP]

r_m, g_m, pairs_m, rho_m = gr_results[TARGET_MLIP]
g_m_sm = gaussian_filter1d(g_m, sigma=2.5)

PAIRS_MLIP = [('C','H'), ('C','O'), ('C','C'), ('O','Zr'), ('Zr','O'), ('Zr','Zr')]
coord_mlip, r_ctr_mlip, el_count_mlip = compute_coordination(data_mlip, PAIRS_MLIP, r_max=5.0, dr=0.02)
for (A, B), res_ in coord_mlip.items():
    print(f'  {A}-{B}: r_peak={res_["r_peak"]:.3f} A   r_min={res_["r_min"]:.3f} A   N_{A}{B}={res_["N_AB"]:.2f}')

PAIR_COLS32 = {
    ('C', 'H') : '#2E7D32', ('C', 'O') : '#FF6F00', ('C', 'C') : '#6A1B9A',
    ('O', 'Zr'): '#0277BD', ('Zr','O') : '#01579B', ('Zr','Zr'): '#B71C1C',
}
PAIR_LABELS32 = {k: f'g({k[0]}–{k[1]})' for k in PAIRS_MLIP}

def _style_ax_wide(ax, fs=12):
    style_ax(ax, fs=fs)
    for sp in ax.spines.values():
        sp.set_linewidth(2.2)

fig = plt.figure(figsize=(13, 17))
gs_outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 2], hspace=0.32)

ax_top = fig.add_subplot(gs_outer[0])
ax_top.plot(r_m, g_m_sm, color=col_mlip, lw=2.6)
ax_top.fill_between(r_m, 0, g_m_sm, alpha=0.13, color=col_mlip)
bold_labels(ax_top, xlabel='r  (Å)', ylabel='g(r)', fs=14)
ax_top.set_xlim(0.8, 7)
ax_top.set_ylim(0, g_m_sm.max()*1.22)
_style_ax_wide(ax_top, fs=13)

gs_inner = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=gs_outer[1], hspace=0.50, wspace=0.38)
X_MAX = 5.0
dr32 = r_ctr_mlip[1] - r_ctr_mlip[0]
i_max = int(X_MAX / dr32)

for idx, key in enumerate(PAIRS_MLIP):
    row, c = divmod(idx, 3)
    ax = fig.add_subplot(gs_inner[row, c])
    if key in coord_mlip:
        g_ab = coord_mlip[key]['g_AB']
        y_max = g_ab[:i_max].max() * 1.30
        ax.plot(r_ctr_mlip, g_ab, color=PAIR_COLS32[key], lw=2.2)
        ax.set_ylim(0, y_max if y_max > 0 else 1)
    bold_labels(ax, xlabel='r  (Å)', ylabel=PAIR_LABELS32[key], fs=12)
    ax.set_xlim(0.5, X_MAX)
    _style_ax_wide(ax, fs=11)

savepath = os.path.join(OUTDIRS[TARGET_MLIP], 'UiO66_MLIP_RDF.png')
plt.savefig(savepath, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', savepath)


---
## 10  MIL-101 (Cr) — Individual Characterization  <a name="10-mil101"></a>

MIL-101(Cr) — Cr₃O(F/OH)(H₂O)₂[C₆H₄(CO₂)₂]₃, UFF force field, 3604 atoms.
XRD / FTIR / BET / PSD / RDF here reuse the results already computed in the
shared section above (`xrd_results`, `ftir_results`, `bet_results`,
`psd_results`, `gr_results` all have a `'MIL-101 (UFF)'` entry) — only the
plotting (with experimental reference panels) and the directed partial-g(r)
coordination numbers are computed fresh.

### Experimental reference extraction

`MIL-101/Data for MIL-101.docx` is an older internal report (~2012) — a zip
archive with pictures under `word/media/`. Of its 16 embedded images, three
are usable experimental reference plots for **pure MIL-101** (the doc also
shows Fe-loaded variants "Fe@M101" / "Fe@M101-TEPA" from a different study —
those curves are visible in the same source image but ignored here):
- Powder XRD (image1.png)
- N₂ physisorption isotherm, used for BET (image2.png)
- FTIR (image3.png)

The other 13 images are instrument photos, thermal-camera stills, and
unrelated plots (isosteric heat, TGA kinetics) — not used. **First-pass
extraction — swap in cleaner source figures if/when available.**


In [ ]:
os.makedirs(EXPDIR['MIL-101 (UFF)'], exist_ok=True)
SRC_DOCX = os.path.join('MIL-101', 'Data for MIL-101.docx')

with zipfile.ZipFile(SRC_DOCX) as z:
    media = sorted(n for n in z.namelist() if n.startswith('word/media/'))
    print(f'{len(media)} embedded images found in the .docx:')
    for m in media:
        print(' ', m)

    # Manually identified by visual inspection (see markdown above):
    #   image1.png -> powder XRD (MIL-101, Fe@M101, Fe@M101-TEPA, standard)
    #   image2.png -> N2 physisorption isotherm (BET)
    #   image3.png -> FTIR transmittance
    KEEP = {
        'word/media/image1.png': 'MIL101_exp_XRD.png',
        'word/media/image2.png': 'MIL101_exp_BET.png',
        'word/media/image3.png': 'MIL101_exp_FTIR.png',
    }
    for src, dst in KEEP.items():
        with z.open(src) as fin, open(os.path.join(EXPDIR['MIL-101 (UFF)'], dst), 'wb') as fout:
            shutil.copyfileobj(fin, fout)
        print(f'extracted {src} -> {os.path.join(EXPDIR["MIL-101 (UFF)"], dst)}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, fname, title in zip(
        axes,
        ['MIL101_exp_XRD.png', 'MIL101_exp_BET.png', 'MIL101_exp_FTIR.png'],
        ['Experimental XRD', 'Experimental N$_2$ isotherm (BET)', 'Experimental FTIR']):
    img = mpimg.imread(os.path.join(EXPDIR['MIL-101 (UFF)'], fname))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')
plt.suptitle('Extracted experimental references — pure MIL-101 curve/trace in each panel\n'
             '(source image also shows Fe-loaded variants from an unrelated study; ignore those)',
             fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
tt, I  = xrd_results[TARGET]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(tt, I, color=col, lw=1.2)
ax.fill_between(tt, 0, I, alpha=0.12, color=col)
pks, _ = find_peaks(I, height=6, distance=12)
for pk in pks[np.argsort(I[pks])[::-1][:8]]:
    ax.annotate(f'{tt[pk]:.2f}°', xy=(tt[pk], I[pk]),
                xytext=(0, 7), textcoords='offset points',
                fontsize=8, ha='center', color=col,
                arrowprops=dict(arrowstyle='-', lw=0.5, color=col))
bold_labels(ax, xlabel='2θ (degrees)', ylabel='Intensity (a.u.)')
ax.set_xlim(2, 45)
ax.set_ylim(-2, max(I.max()*1.25, 20))
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'MIL101_XRD.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
wn, spec, ir_peaks = ftir_results[TARGET]
spec_sm = gaussian_filter1d(spec, sigma=3)
trans   = 100.0 - spec_sm

IR_BANDS_MIL = [
    (400,  700,  'M–O / M–F', '#FFCDD2'),
    (700,  900,  'C–H bend',  '#C8E6C9'),
    (1000, 1200, 'C–O str.',  '#BBDEFB'),
    (1350, 1650, 'COO⁻',      '#FFE0B2'),
    (2850, 3050, 'C–H str.',  '#B2EBF2'),
    (3200, 3700, 'O–H str.',  '#FFF9C4'),
]

fig, ax = plt.subplots(figsize=(9, 5.5))
for x0, x1, lbl, rcol in IR_BANDS_MIL:
    ax.axvspan(x0, x1, alpha=0.15, color=rcol, zorder=0)
    ax.text((x0+x1)/2, 102, lbl, ha='center', va='bottom', fontsize=8, color='#555')
ax.plot(wn, trans, color=col, lw=1.8, zorder=5)
ax.fill_between(wn, 100, trans, alpha=0.13, color=col)
pidx, _ = find_peaks(spec_sm, height=12, distance=25)
for pk in pidx[np.argsort(spec_sm[pidx])[::-1][:8]]:
    ax.annotate(f'{wn[pk]:.0f}', xy=(wn[pk], trans[pk]),
                xytext=(4, 6), textcoords='offset points',
                fontsize=8.5, color=col, rotation=45, ha='left', fontweight='bold')
bold_labels(ax, xlabel='Wavenumber  (cm⁻¹)', ylabel='Transmittance (a.u.)')
ax.set_xlim(4000, 400)
ax.set_ylim(0, 108)
ax.set_yticks([])
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'MIL101_FTIR.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
b      = bet_results[TARGET]

fig, ax = plt.subplots(figsize=(7, 5.5))
categories = ['SA\n(m²/g)', 'Geometric\nvoid (%)', 'N₂-accessible\nvoid (%)']
values = [b['sa_m2_per_g'], b['void_geometric']*100, b['void_accessible']*100]
bar_colors = ['#42A5F5', '#66BB6A', '#FFA726']
bars = ax.bar(categories, values, color=bar_colors, edgecolor='k', lw=0.7, alpha=0.88, width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(values)*0.02,
            f'{v:.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(3500, color='#42A5F5', lw=1.2, ls='--', alpha=0.6, label='Lit. SA ~3500-4500 m$^2$/g')
ax.axhline(80,   color='#66BB6A', lw=1.2, ls='--', alpha=0.6, label='Lit. geom. void ~80-85%')
bold_labels(ax, ylabel='Value')
ax.set_ylim(0, max(values)*1.35)
ax.set_xticklabels(categories, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'MIL101_BET.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
res    = psd_results[TARGET]
r      = res['r']
psd_sm  = gaussian_filter1d(res['psd_vol'],   sigma=2)
psdc_sm = gaussian_filter1d(res['psd_count'], sigma=2)
ymax = psd_sm.max()
di, dpk, d50 = res['d_i'], res['d_peak_vol'], res['d_50']

MOLECULES_M = {'H$_2$O': 2.65, 'H$_2$': 2.89, 'CO$_2$': 3.30, 'O$_2$': 3.46,
               'N$_2$': 3.64, 'CH$_4$': 3.80, 'SF$_6$': 5.50, 'Benzene': 5.85}
MOL_COLS_M = ['#6A1B9A','#1565C0','#B71C1C','#00838F','#2E7D32','#E65100','#F9A825','#4E342E']

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r*2, psd_sm,  color=col, lw=2.2, label='Volume-weighted')
ax.plot(r*2, psdc_sm, color=col, lw=1.0, ls='--', alpha=0.45, label='Count-weighted')
ax.fill_between(r*2, 0, psd_sm, alpha=0.12, color=col)
ax.axvline(di,  color=col, lw=1.8, ls='-.', label=f'$D_i$ = {di:.1f} Å')
ax.axvline(dpk, color=col, lw=1.3, ls=':',  label=f'$D_{{peak}}$ = {dpk:.1f} Å')
ax.axvspan(0, d50, alpha=0.05, color=col)
for (mol, kd), mcol in zip(MOLECULES_M.items(), MOL_COLS_M):
    if kd < r[-1]*2:
        ax.axvline(kd, color=mcol, lw=1.0, ls=':', alpha=0.7)
        ax.text(kd+0.1, ymax*0.98, mol, fontsize=7.5, color=mcol, va='top', rotation=90, alpha=0.9)
bold_labels(ax, xlabel='Pore diameter  (Å)', ylabel='PSD  (normalised)')
ax.set_xlim(0, min(r[-1]*2, 40))
ax.set_ylim(0, ymax*1.35)
ax.legend(fontsize=9, loc='upper right', ncol=2)
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'MIL101_PSD.png')
plt.savefig(savepath, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", savepath)


In [ ]:
TARGET = 'MIL-101 (UFF)'
data_mil = structures[TARGET]
col_mil  = COLORS[TARGET]

r_mil, g_mil, pairs_mil, rho_mil = gr_results[TARGET]
g_mil_sm = gaussian_filter1d(g_mil, sigma=2.5)

PLOT_PAIRS_MIL = [('C','H'), ('C','O'), ('C','C'), ('O','Cr'), ('Cr','O'), ('Cr','Cr')]
coord_mil, r_ctr_mil, el_count_mil = compute_coordination(data_mil, PLOT_PAIRS_MIL, r_max=5.0, dr=0.02)
for (A, B), res_ in coord_mil.items():
    print(f'  {A}-{B}: r_peak={res_["r_peak"]:.3f} A   r_min={res_["r_min"]:.3f} A   N_{A}{B}={res_["N_AB"]:.2f}')

PAIR_COLS_MIL = {('C','H'):'#2E7D32', ('C','O'):'#FF6F00', ('C','C'):'#6A1B9A',
                  ('O','Cr'):'#0277BD', ('Cr','O'):'#01579B', ('Cr','Cr'):'#B71C1C'}
PAIR_LABELS_MIL = {k: f'g({k[0]}–{k[1]})' for k in PLOT_PAIRS_MIL}

def _style_ax_wide_mil(ax, fs=12):
    style_ax(ax, fs=fs)
    for sp in ax.spines.values():
        sp.set_linewidth(2.2)

fig = plt.figure(figsize=(13, 17))
gs_outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 2], hspace=0.32)

ax_top = fig.add_subplot(gs_outer[0])
ax_top.plot(r_mil, g_mil_sm, color=col_mil, lw=2.6)
ax_top.fill_between(r_mil, 0, g_mil_sm, alpha=0.13, color=col_mil)
bold_labels(ax_top, xlabel='r  (Å)', ylabel='g(r)', fs=14)
ax_top.set_xlim(0.8, 12)
ax_top.set_ylim(0, g_mil_sm.max()*1.22)
_style_ax_wide_mil(ax_top, fs=13)

gs_inner = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=gs_outer[1], hspace=0.50, wspace=0.38)
X_MAX = 5.0
dr32 = r_ctr_mil[1] - r_ctr_mil[0]
i_max = int(X_MAX / dr32)

for idx, key in enumerate(PLOT_PAIRS_MIL):
    row, c = divmod(idx, 3)
    ax = fig.add_subplot(gs_inner[row, c])
    if key in coord_mil:
        g_ab = coord_mil[key]['g_AB']
        y_max = g_ab[:i_max].max() * 1.30
        ax.plot(r_ctr_mil, g_ab, color=PAIR_COLS_MIL[key], lw=2.2)
        ax.set_ylim(0, y_max if y_max > 0 else 1)
    bold_labels(ax, xlabel='r  (Å)', ylabel=PAIR_LABELS_MIL[key], fs=12)
    ax.set_xlim(0.5, X_MAX)
    _style_ax_wide_mil(ax, fs=11)

savepath = os.path.join(OUTDIRS[TARGET], 'MIL101_RDF.png')
plt.savefig(savepath, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', savepath)


---
## 11  Final Manuscript Figures  <a name="11-final-figures"></a>

Clean, unannotated versions of the key comparison plots for the write-up:
no titles, no in-plot peak/value labels, no shaded band highlights, bold
axis labels, frameless legends. All reuse results already computed above.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 1 — XRD of UiO-66 (MACE)
# ══════════════════════════════════════════════════════════════════════════════
TARGET = 'UiO-66 (MLIP)'
col    = COLORS['UiO-66 (MACE)']
tt, I  = xrd_results[TARGET]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(tt, I, color=col, lw=1.4)
ax.fill_between(tt, 0, I, alpha=0.12, color=col)
bold_labels(ax, xlabel='2θ (degrees)', ylabel='Intensity (a.u.)')
ax.set_xlim(2, 45)
ax.set_ylim(-2, max(I.max()*1.15, 20))
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'Fig1_UiO66_MACE_XRD.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 2 — RDF of UiO-66 (MACE)
# ══════════════════════════════════════════════════════════════════════════════
def _style_wide(ax):
    style_ax(ax)
    for sp in ax.spines.values():
        sp.set_linewidth(2.2)

TARGET_MLIP = 'UiO-66 (MLIP)'
col_mace    = COLORS['UiO-66 (MACE)']

r_m, g_m, pairs_m, rho_m = gr_results[TARGET_MLIP]
g_m_sm = gaussian_filter1d(g_m, sigma=2.5)

fig = plt.figure(figsize=(13, 17))
gs_outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 2], hspace=0.32)

ax_top = fig.add_subplot(gs_outer[0])
ax_top.plot(r_m, g_m_sm, color=col_mace, lw=2.6)
ax_top.fill_between(r_m, 0, g_m_sm, alpha=0.13, color=col_mace)
bold_labels(ax_top, xlabel='r  (Å)', ylabel='g(r)', fs=14)
ax_top.set_xlim(0.8, 7)
ax_top.set_ylim(0, g_m_sm.max()*1.22)
_style_wide(ax_top)

gs_inner = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=gs_outer[1], hspace=0.50, wspace=0.38)
X_MAX = 5.0
dr32  = r_ctr_mlip[1] - r_ctr_mlip[0]
i_max = int(X_MAX / dr32)

for idx, key in enumerate(PAIRS_MLIP):
    row, c = divmod(idx, 3)
    ax = fig.add_subplot(gs_inner[row, c])
    if key in coord_mlip:
        g_ab = coord_mlip[key]['g_AB']
        y_max = g_ab[:i_max].max() * 1.30
        ax.plot(r_ctr_mlip, g_ab, color=PAIR_COLS32[key], lw=2.2)
        ax.set_ylim(0, y_max if y_max > 0 else 1)
    bold_labels(ax, xlabel='r  (Å)', ylabel=PAIR_LABELS32[key], fs=12)
    ax.set_xlim(0.5, X_MAX)
    _style_wide(ax)

savepath = os.path.join(OUTDIRS[TARGET_MLIP], 'Fig2_UiO66_MACE_RDF.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 3 — FTIR of UiO-66 (MACE)
# ══════════════════════════════════════════════════════════════════════════════
TARGET = 'UiO-66 (MACE)'

fig, ax = plt.subplots(figsize=(8, 5.5))
img_mace = mpimg.imread(MACE_FTIR_PNG)
ax.imshow(img_mace); ax.axis('off')

plt.tight_layout()
savepath = os.path.join(OUTDIRS['UiO-66 (MLIP)'], 'Fig3_UiO66_MACE_FTIR.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 4 — PSD of UiO-66: UFF vs MACE
# ══════════════════════════════════════════════════════════════════════════════
res_uff  = psd_results['UiO-66 (UFF)']
res_mace = psd_results['UiO-66 (MLIP)']
col_uff  = COLORS['UiO-66 (UFF)']
col_mace = COLORS['UiO-66 (MACE)']

psd_uff_sm  = gaussian_filter1d(res_uff['psd_vol'],  sigma=2)
psd_mace_sm = gaussian_filter1d(res_mace['psd_vol'], sigma=2)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(res_uff['r']*2,  psd_uff_sm,  color=col_uff,  lw=2.2, label='UFF')
ax.plot(res_mace['r']*2, psd_mace_sm, color=col_mace, lw=2.2, label='MACE')
bold_labels(ax, xlabel='Pore diameter  (Å)', ylabel='Probability', fs=12)
ax.set_xlim(0, 10)
ax.set_ylim(0, max(psd_uff_sm.max(), psd_mace_sm.max())*1.2)
ax.legend(loc='upper right')
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS['UiO-66 (UFF)'], 'Fig4_UiO66_PSD_UFF_vs_MACE.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 5 — XRD of MIL-101 (computed)
# ══════════════════════════════════════════════════════════════════════════════
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
tt, I  = xrd_results[TARGET]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(tt, I, color=col, lw=1.4)
ax.fill_between(tt, 0, I, alpha=0.12, color=col)
bold_labels(ax, xlabel='2θ (degrees)', ylabel='Intensity (a.u.)')
ax.set_xlim(2, 45)
ax.set_ylim(-2, max(I.max()*1.15, 20))
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'Fig5_MIL101_XRD.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 6 — RDF of MIL-101
# ══════════════════════════════════════════════════════════════════════════════
TARGET_MIL = 'MIL-101 (UFF)'
col_mil    = COLORS[TARGET_MIL]

r_mil, g_mil, pairs_mil, rho_mil = gr_results[TARGET_MIL]
g_mil_sm = gaussian_filter1d(g_mil, sigma=2.5)

fig = plt.figure(figsize=(13, 17))
gs_outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 2], hspace=0.32)

ax_top = fig.add_subplot(gs_outer[0])
ax_top.plot(r_mil, g_mil_sm, color=col_mil, lw=2.6)
ax_top.fill_between(r_mil, 0, g_mil_sm, alpha=0.13, color=col_mil)
bold_labels(ax_top, xlabel='r  (Å)', ylabel='g(r)', fs=14)
ax_top.set_xlim(0.8, 12)
ax_top.set_ylim(0, g_mil_sm.max()*1.22)
_style_wide(ax_top)

gs_inner = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=gs_outer[1], hspace=0.50, wspace=0.38)
X_MAX = 5.0
dr32  = r_ctr_mil[1] - r_ctr_mil[0]
i_max = int(X_MAX / dr32)

for idx, key in enumerate(PLOT_PAIRS_MIL):
    row, c = divmod(idx, 3)
    ax = fig.add_subplot(gs_inner[row, c])
    if key in coord_mil:
        g_ab = coord_mil[key]['g_AB']
        y_max = g_ab[:i_max].max() * 1.30
        ax.plot(r_ctr_mil, g_ab, color=PAIR_COLS_MIL[key], lw=2.2)
        ax.set_ylim(0, y_max if y_max > 0 else 1)
    bold_labels(ax, xlabel='r  (Å)', ylabel=PAIR_LABELS_MIL[key], fs=12)
    ax.set_xlim(0.5, X_MAX)
    _style_wide(ax)

savepath = os.path.join(OUTDIRS[TARGET_MIL], 'Fig6_MIL101_RDF.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 7 — FTIR of MIL-101
# ══════════════════════════════════════════════════════════════════════════════
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
wn, spec, ir_peaks = ftir_results[TARGET]
spec_sm = gaussian_filter1d(spec, sigma=3)
trans   = 100.0 - spec_sm

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(wn, trans, color=col, lw=1.8)
bold_labels(ax, xlabel='Wavenumber  (cm⁻¹)', ylabel='Transmittance (a.u.)')
ax.set_xlim(4000, 400)
ax.set_ylim(0, 105)
ax.set_yticks([])
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'Fig7_MIL101_FTIR.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig. 8 — PSD of MIL-101
# ══════════════════════════════════════════════════════════════════════════════
TARGET = 'MIL-101 (UFF)'
col    = COLORS[TARGET]
res    = psd_results[TARGET]
psd_sm  = gaussian_filter1d(res['psd_vol'],   sigma=2)
psdc_sm = gaussian_filter1d(res['psd_count'], sigma=2)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(res['r']*2, psd_sm,  color=col, lw=2.2, label='Volume-weighted')
#ax.plot(res['r']*2, psdc_sm, color=col, lw=1.0, ls='--', alpha=0.5, label='Count-weighted')
bold_labels(ax, xlabel='Pore diameter  (Å)', ylabel='Probability', fs=12)
ax.set_xlim(0, min(res['r'][-1]*2, 40))
ax.set_ylim(0, psd_sm.max()*1.2)
ax.legend(loc='upper right')
style_ax(ax)

plt.tight_layout()
savepath = os.path.join(OUTDIRS[TARGET], 'Fig8_MIL101_PSD.png')
plt.savefig(savepath, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', savepath)